# Monthly Performance Report — OLJ + OT, **Matomo-based**

One notebook builds **both** Matomo monthly reports — the parallel of the GA4 report
(`monthly_report_GA4.ipynb`), same layout, in Matomo's words (visits, unique visitors, pageviews…). For each brand it
produces and emails:

- the Word report — `OLJ.September.2026.Matomo.docx`
- an Excel file of **every country's visits and share of visits** — `OLJ.September.2026.Matomo.Countries.xlsx`

| Step | What it does | Source |
|---|---|---|
| ⚙️ Settings | Matomo, CMS key, acquisitions sheet, brands, recipients | — |
| 📅 Month | **last month by default**; one line to uncomment to pick any month | — |
| 1. New accounts | accounts created per month, per brand | CMS |
| 2. Acquisitions | new subscriptions per month, per brand | acquisitions Google Sheet |
| 3. Matomo | visits, pageviews, unique visitors, visit duration, channel types, countries | Matomo |
| 3b. Countries | this month's top 5 + Others, 6.3 for countries that left the top 5 | Matomo |
| 4. GA4 — 2 sections | **section 5** distribution channels and **section 7** top articles + top 2 sources, **marked \* and "GA4 data"** in the report | GA4 |
| 5–8 | brands, charts, Word, **Excel (countries)**, email | — |

Run top to bottom. On the 1st of the month it builds the month that just ended.

## Dependencies

Run this once per Colab session (installs are wiped when the runtime resets).

In [ ]:
!pip install -q google-analytics-data google-api-python-client google-auth google-auth-oauthlib python-docx openpyxl requests matplotlib

## ⚙️ Settings

**Matomo** (`analytics-cms.whitebeard.me`, site 2) is the main source; GA4 is used only for sections 5 and 7 (marked * in the report). Each brand is one of Matomo's
**pre-processed** segments on `dimension5` (website + iOS + Android app IDs), in `MATOMO_SEGMENTS` — written exactly as
Matomo stores it. The token comes from the GitHub secret / Colab 🔑 secret `MATOMO_TOKEN` — never paste it in the notebook.

The CMS key (`CMS_API_KEY`) and `service_account.json` (the acquisitions sheet) work exactly as in the GA4 report.

**Cover logos:** stored in the 🎨 Brand assets cell (or `olj_logo.png` / `ot_logo.png` next to the notebook).

In [ ]:
import os

# ---- CMS (new accounts, both brands) ---------------------------------------------------------------------------------
CMS_API_KEY = ""   # <-- paste your CMS API key between the quotes (leave EMPTY if this notebook goes to GitHub)

def _colab_secret(name):
    """Colab's 🔑 Secrets panel (left sidebar) -- survives new copies of the notebook, unlike a pasted key."""
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

# Where the key comes from, first match wins: 1. GitHub secret / env var  2. Colab secret  3. pasted above
_pasted_key = CMS_API_KEY.strip()
for _src, _val in (("GitHub secret / env var", os.environ.get("CMS_API_KEY", "")),
                   ("Colab secret", _colab_secret("CMS_API_KEY")),
                   ("pasted in this cell", _pasted_key)):
    if _val.strip():
        CMS_API_KEY, CMS_KEY_SOURCE = _val.strip(), _src
        break
else:
    CMS_API_KEY, CMS_KEY_SOURCE = "", "none"

if os.environ.get("GITHUB_ACTIONS") == "true" and _pasted_key:
    print("⚠️  A CMS key is pasted in this notebook and it's in the GitHub repo -- delete it from the cell and "
          "rotate the key in the CMS backend; the GitHub secret is all the workflow needs.")

CMS_API_HOST = "api.lorientlejour.com"
CMS_API_HOST = (CMS_API_HOST or os.environ.get("CMS_API_HOST", "")).strip().removeprefix("https://").removeprefix("http://").strip("/")
CMS_BASE_URL = f"https://{CMS_API_HOST}" if CMS_API_HOST else ""

# Same backend URL as the weekly brief; {start} / {end} are filled in with the first and last day of the 6 months.
CMS_BACKEND_URL = (
    "https://managecmsnew.lorientlejour.com/revenue/customer?"
    "columns%5B%5D=id&columns%5B%5D=creationDate&columns%5B%5D=preferredLanguage&columns%5B%5D=source"
    "&creationDate_operator=between&creationDate%5Bstart%5D={start}&creationDate%5Bend%5D={end}"
)
CMS_DATE_FMT = "%Y-%m-%d"

# ---- Acquisitions sheet (the ONLY source for acquisitions) -------------------------------------------------------------
# Read with a Google service account (service_account.json). Its e-mail needs Viewer access to the sheet.
ACQ_SHEET_ID = "11WU-b3nmvyPlO0fX9VycKgObr-v5-hXTN6ieCv2TOoA"
ACQ_SHEET_GID = 2059832827          # the tab from the link (#gid=...); falls back to a tab named "Acquisitions"
ACQ_SHEET_XLSX = "Daily_sheet_Acquisitions_Churns.xlsx"   # optional fallback: the sheet downloaded and uploaded in Colab

# ---- What to build -------------------------------------------------------------------------------------------------------
BUILD_BRANDS = ["OLJ", "OT"]      # remove one to build a single report
N_MONTHS = 6                      # months shown in the charts and the table (the report month + the 5 before it)

# ---- Cover branding --------------------------------------------------------------------------------------------------------
# Logo on the cover and in the page header, per brand (PNG or JPG -- Word can't embed SVG). First match wins:
#   1. a file next to the notebook / uploaded in Colab, named as in LOGO_FILES (to replace a logo, just drop the file in)
#   2. a direct link to the image in LOGO_URLS (downloaded at run time)
#   3. the logo stored in this notebook (🎨 Brand assets cell below -- OLJ's is already there)
# With none of them, the cover prints the brand name in HEADLINE_FONT instead.
LOGO_FILES = {"OLJ": "olj_logo.png", "OT": "ot_logo.png"}
LOGO_URLS = {
    "OLJ": "",   # <-- e.g. the logo's image link from the site (right-click the logo -> Copy image address)
    "OT":  "",
}
BRAND_ACCENT = {"OLJ": "1B1B1B", "OT": "1B1B1B"}   # accent on the cover rule, labels and section openers (the logo's black)
DATA_SOURCE = "Matomo"   # this notebook = monthly_report_matomo.ipynb (the GA4 report is monthly_report_GA4.ipynb)

# ---- Matomo: every number in the report ----------------------------------------------------------------------------------
MATOMO_URL = "https://analytics-cms.whitebeard.me"
MATOMO_SITE_ID = 2
MATOMO_TOKEN = ""   # leave EMPTY: read from the GitHub secret / env var MATOMO_TOKEN or the Colab 🔑 secret MATOMO_TOKEN
MATOMO_TOKEN = (os.environ.get("MATOMO_TOKEN") or _colab_secret("MATOMO_TOKEN") or MATOMO_TOKEN).strip()
# Each brand = a PRE-PROCESSED Matomo segment. Write it EXACTLY as Matomo stores it (the "segment=" part of a Matomo
# API link, decoded once): Matomo only uses the pre-computed numbers when the definition matches character for character
# -- e.g. the ":" in the app IDs is "%253A" here. Any other spelling = computed live = slow (Cloudflare 524 timeouts).
MATOMO_SEGMENT_DIMENSION = "dimension5"
MATOMO_SEGMENTS = {
    "OLJ": "dimension5==www.lorientlejour.com,dimension5==ios%253Ame.whitebeard.olj,dimension5==android%253Ame.whitebeard.olj",
    "OT":  "dimension5==today.lorientlejour.com,dimension5==ios%253Ame.whitebeard.olj.today,"
           "dimension5==android%253Ame.whitebeard.olj.today",
}

REPORT_NAME = "Monthly Performance Report (Matomo)"   # cover, page header, document properties, email
REPORT_SOURCE = "Matomo"                              # last part of the file name: OLJ.August.2026.Matomo.docx
HEADLINE_FONT = "Georgia"   # headlines / titles. To use the brand font, type its exact name here -- it must be installed
                            # on the computers that open the report (Word substitutes a default font otherwise)

# ---- Optional disclaimer under the traffic charts (leave "" for none) ------------------------------------------------------
DISCLAIMERS = {
    "OLJ": "",
    "OT":  "",
}

# ---- Email recipients -----------------------------------------------------------------------------------------------------
# Test runs (you in Colab, or a manual "Run workflow" on GitHub) only go to TEST_RECIPIENTS -- both reports.
TEST_RECIPIENTS = ["dianafarhat@lorientlejour.com"]
# The scheduled run on the 1st goes to each brand's list, or to the GitHub variables
# MONTHLY_RECIPIENTS_OLJ / MONTHLY_RECIPIENTS_OT (comma-separated) when they're set.
SCHEDULED_RECIPIENTS = {
    "OLJ": ["dianafarhat@lorientlejour.com"],
    "OT":  ["dianafarhat@lorientlejour.com"],
}
EMAIL_AS_BCC = False   # True = recipients don't see each other (and can't reply-all)

print("Matomo:", MATOMO_URL, f"site {MATOMO_SITE_ID}", "| token:", "set" if MATOMO_TOKEN else "MISSING")
print("CMS key:", f"set (from {CMS_KEY_SOURCE})" if CMS_API_KEY else "MISSING", "| API:", (CMS_BASE_URL + "/cms/customer") if CMS_BASE_URL else "host MISSING")

## 🎨 Brand assets

Both logos stored inside the notebook (OLJ and OT, transparent background), so the report has them in Colab and on GitHub without uploading anything. To replace one, drop `olj_logo.png` / `ot_logo.png` next to the notebook.

In [ ]:
# base64-encoded PNGs (transparent background). Leave a brand out to fall back to LOGO_FILES / LOGO_URLS / text.
LOGO_EMBEDDED = {
    "OLJ": "...",
    "OT": "...",
}

## 📅 Which month

**By default it's last month**, so a run on 1 October builds September. To build any other month, uncomment
the override line and type the year and month. The report shows that month and the 5 months before it.

In [ ]:
import calendar
import datetime as dt
import pandas as pd

# ╔══════════════════════════════════════════════════════════════════════════════════════════════╗
# ║  DEFAULT: last month (the month that just ended).                                                 ║
REPORT_MONTH = (dt.date.today().replace(day=1) - dt.timedelta(days=1)).replace(day=1)
# ║  OVERRIDE: to pick any month, remove the # at the start of the next line and edit (year, month, 1).  ║
# REPORT_MONTH = dt.date(2026, 8, 1)      # <- e.g. dt.date(2026, 8, 1) = the August 2026 report
# ╚══════════════════════════════════════════════════════════════════════════════════════════════╝

REPORT_MONTH = REPORT_MONTH.replace(day=1)

def _shift_month(d, n):
    y, m = divmod(d.month - 1 + n, 12)
    return dt.date(d.year + y, m + 1, 1)

def _month_end(d):
    return d.replace(day=calendar.monthrange(d.year, d.month)[1])

MONTHS = [_shift_month(REPORT_MONTH, -i) for i in range(N_MONTHS - 1, -1, -1)]   # oldest -> report month
MONTH_LABELS = [f"{m:%Y-%m}" for m in MONTHS]                                     # "2026-08" (chart axis labels)
WINDOW_START = dt.datetime.combine(MONTHS[0], dt.time.min)
WINDOW_END = dt.datetime.combine(_month_end(REPORT_MONTH), dt.time.min)
PREV_MONTH = MONTHS[-2] if len(MONTHS) > 1 else None
PERIOD_LINE = f"Period: {MONTHS[0]:%b %Y} – {REPORT_MONTH:%b %Y}"

if WINDOW_END.date() >= dt.date.today():
    print(f"⚠️  {REPORT_MONTH:%B %Y} isn't over yet -- its numbers will be partial.")
print(f"Report month: {REPORT_MONTH:%B %Y}   |   months shown: {', '.join(MONTH_LABELS)}")

## 1. New accounts — CMS (OLJ and OT)

Exactly the weekly brief's logic, bucketed by **month** instead of week:

| Field | Rule |
|---|---|
| **Brand** (`preferredLanguage`) | `en` / `english` / `anglais` → **OT** · anything else → **OLJ** |
| **Platform** (`source`) | `web_olj`, `web_ot` → Web · `app`, `todayapp` → App · anything else → Other |

The report's "Accounts created" is the **Total** (Web + App + Other). `creationDate` is UTC, so it's converted to
Beirut time before bucketing. If the CMS isn't configured or the call fails, the cell stops with an error.

In [ ]:
import requests
from collections import Counter
from urllib.parse import urlsplit, parse_qsl
from zoneinfo import ZoneInfo

OT_LANGUAGES = {"en", "eng", "english", "anglais"}
SOURCE_PLATFORM = {"web_olj": "Web", "web_ot": "Web", "app": "App", "todayapp": "App"}
OTHER_BUCKET = "Other"
ACCOUNT_BUCKETS = ["Web", "App", OTHER_BUCKET]

CMS_TZ = ZoneInfo("Asia/Beirut")
CMS_MAX_PAGES = 5000        # safety stop (6 months is more pages than a week)
CMS_READY = bool(CMS_API_KEY and CMS_BASE_URL)

def _cms_check(resp):
    if resp.status_code in (403, 503) and "just a moment" in resp.text[:2000].lower():
        raise RuntimeError(
            "[CMS] Blocked by Cloudflare's bot check, not by the CMS -- the API key is fine.\n"
            "  Try Runtime -> Disconnect and delete runtime, then rerun.")
    resp.raise_for_status()

def _pick(c, *names):
    wanted = {n.lower() for n in names}
    for scope in (c, c.get("fields"), c.get("preferences")):
        if isinstance(scope, dict):
            for k, v in scope.items():
                if k.lower() in wanted and v not in (None, ""):
                    return v
    return None

def _lang(c):
    return str(_pick(c, "preferredLanguage", "preferred_language", "language") or "").strip().lower()

def _source(c):
    return str(_pick(c, "source", "acquisitionSource", "acquisition_source") or "").strip().lower()

def brand_of(c):
    lang = _lang(c)
    return "ot" if lang in OT_LANGUAGES or lang.startswith(("en-", "en_")) else "olj"

def platform_of(c):
    return SOURCE_PLATFORM.get(_source(c), OTHER_BUCKET)

def _created_local(c):
    raw = _pick(c, "creationDate", "creation_date", "created")
    if not raw:
        return None
    try:
        d = dt.datetime.fromisoformat(str(raw).replace("Z", "+00:00"))
    except ValueError:
        return None
    if d.tzinfo is None:
        d = d.replace(tzinfo=dt.timezone.utc)
    return d.astimezone(CMS_TZ).replace(tzinfo=None)

def _backend_params(start, end):
    query = urlsplit(CMS_BACKEND_URL).query if "?" in CMS_BACKEND_URL else CMS_BACKEND_URL
    fill = lambda v: v.replace("{start}", start.strftime(CMS_DATE_FMT)).replace("{end}", end.strftime(CMS_DATE_FMT))
    params = [(k, fill(v)) for k, v in parse_qsl(query, keep_blank_values=True) if k != "page" and v != ""]
    has_date_filter = any("{start}" in v for _, v in parse_qsl(query))
    return params, has_date_filter

def cms_fetch_customers(start, end):
    """Every customer the query returns, across all pages (deduped by id)."""
    base_params, has_date_filter = _backend_params(start, end)
    if not has_date_filter:
        print("[CMS] no creation-date filter in CMS_BACKEND_URL -- fetching and filtering here (slower)")
    session = requests.Session()
    session.headers.update({"API-Key": CMS_API_KEY, "Accept": "application/json"})
    url = CMS_BASE_URL.rstrip("/") + "/cms/customer"

    customers, seen, page = [], set(), None
    newest_first, prev_last = True, None
    for _ in range(CMS_MAX_PAGES):
        params = base_params + ([("page", page)] if page is not None else [])
        resp = session.get(url, params=params, timeout=60)
        _cms_check(resp)
        payload = resp.json()
        batch = []
        for c in payload.get("data") or []:
            key = str(_pick(c, "userId", "id") or id(c))
            if key not in seen:
                seen.add(key); batch.append(c)
        if not batch:
            break
        customers += batch
        total = payload.get("total") or 0
        if total and len(seen) >= total:
            break
        dates = [d for d in map(_created_local, batch) if d]
        if dates:
            chain = ([prev_last] if prev_last else []) + dates
            newest_first = newest_first and all(a >= b for a, b in zip(chain, chain[1:]))
            prev_last = dates[-1]
            if not has_date_filter and newest_first and dates[-1] < start:
                break
        page = (payload["page"] if payload.get("page") is not None else (page or 0)) + 1
    else:
        print(f"[CMS] stopped after {CMS_MAX_PAGES} pages -- add the creation-date filter to CMS_BACKEND_URL")
    return customers

def cms_accounts_by_month(customers):
    """{'olj': DataFrame, 'ot': DataFrame} -- one row per month (MONTH_LABELS), columns Web / App / Other / Total."""
    counts = {b: {lab: Counter() for lab in MONTH_LABELS} for b in ("olj", "ot")}
    for c in customers:
        d = _created_local(c)
        if d is not None and WINDOW_START.date() <= d.date() <= WINDOW_END.date():
            counts[brand_of(c)][f"{d:%Y-%m}"][platform_of(c)] += 1
    out = {}
    for b, months in counts.items():
        df = pd.DataFrame([{k: months[lab].get(k, 0) for k in ACCOUNT_BUCKETS} for lab in MONTH_LABELS], index=MONTH_LABELS)
        df["Total"] = df[ACCOUNT_BUCKETS].sum(axis=1)
        out[b] = df
    return out

if not CMS_READY:
    raise RuntimeError("CMS not configured -- fill in CMS_API_KEY in ⚙️ Settings (or the Colab / GitHub secret).")
try:
    # starts a day early: the CMS filters by UTC date, we bucket in Beirut time (the local check drops the extra day)
    _customers = cms_fetch_customers(WINDOW_START - dt.timedelta(days=1), WINDOW_END)
except requests.HTTPError as e:
    raise RuntimeError(f"[CMS] new accounts failed (401/403 -> check CMS_API_KEY): {e}") from e
ACCOUNTS = cms_accounts_by_month(_customers)

_in_window = [c for c in _customers if (d := _created_local(c)) and WINDOW_START.date() <= d.date() <= WINDOW_END.date()]
print(f"[CMS] {len(_customers)} fetched, {len(_in_window)} created {MONTHS[0]:%b %Y} – {REPORT_MONTH:%b %Y}")
print("  source values:  ", dict(Counter(_source(c) or "(blank)" for c in _in_window).most_common()))
print("  language values:", dict(Counter(_lang(c) or "(blank)" for c in _in_window).most_common()))
for _b in ("olj", "ot"):
    print(f"\n{_b.upper()} accounts created:\n{ACCOUNTS[_b].to_string()}")

## 2. Acquisitions — from the acquisitions sheet only

Reads the daily *Acquisitions* tab (same as the weekly brief's ( ) figures) and sums the days of each month:
**OLJ = Totales Basic + Totales Premium**, **OT = Totales OT**. Nothing comes from the CMS here.

Days nobody has filled in yet are skipped and reported. If the report month isn't complete in the sheet
(e.g. the last day is typed in the morning after), the Word report says so under the chart.
If the sheet can't be read at all, the cell stops with an error rather than reporting zeros.

In [ ]:
from pathlib import Path

def _parse_acq_sheet(raw):
    """raw: the tab as a DataFrame (no header). -> {'YYYY-MM-DD': {'olj': Basic + Premium, 'ot': OT}} for filled-in days."""
    head = raw.head(5).astype(str).apply(lambda col: col.str.strip().str.lower())
    def col_of(text):
        hits = [c for c in head.columns if head[c].str.contains(text, regex=False).any()]
        if not hits:
            raise ValueError(f"no '{text}' column")
        return hits[0]
    c_date, c_basic, c_prem, c_ot = col_of("date"), col_of("totales basic"), col_of("totales premium"), col_of("totales ot")
    detail = [c for c in raw.columns if c_date < c < min(c_basic, c_prem, c_ot)
              and not head[c].str.contains("total|acquisitions", regex=True).any()]
    num = lambda v: 0 if pd.isna(pd.to_numeric(v, errors="coerce")) else int(float(v))
    out = {}
    for _, r in raw.iterrows():
        v = r[c_date]
        d = (pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(v))) if isinstance(v, (int, float)) and not pd.isna(v) \
            else pd.to_datetime(v, errors="coerce")
        if pd.isna(d) or all(pd.isna(pd.to_numeric(r[c], errors="coerce")) for c in detail):
            continue                                    # not a date, or a day nobody has filled in yet
        out[d.strftime("%Y-%m-%d")] = {"olj": num(r[c_basic]) + num(r[c_prem]), "ot": num(r[c_ot])}
    return out

def read_acq_sheet():
    """Daily totals from the acquisitions sheet; {} (and a printed reason) if it can't be read."""
    try:
        from google.oauth2 import service_account as _sa
        from googleapiclient.discovery import build as _build
        creds = _sa.Credentials.from_service_account_file(
            "service_account.json", scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"])
        api = _build("sheets", "v4", credentials=creds, cache_discovery=False).spreadsheets()
        tabs = [t["properties"] for t in api.get(spreadsheetId=ACQ_SHEET_ID, fields="sheets.properties").execute()["sheets"]]
        order = [t["title"] for t in tabs if t["sheetId"] == ACQ_SHEET_GID] + \
                [t["title"] for t in tabs if t["title"].strip().lower() == "acquisitions"]
        for title in dict.fromkeys(order):
            rows = api.values().get(spreadsheetId=ACQ_SHEET_ID, range=f"'{title}'", valueRenderOption="UNFORMATTED_VALUE",
                                    dateTimeRenderOption="SERIAL_NUMBER").execute().get("values", [])
            width = max((len(r) for r in rows), default=0)
            try:
                daily = _parse_acq_sheet(pd.DataFrame([r + [None] * (width - len(r)) for r in rows]))
                print(f"[sheet] read '{title}' from the Google Sheet: {len(daily)} filled-in days")
                return daily
            except ValueError as e:
                print(f"[sheet] tab '{title}' doesn't look like the acquisitions tab ({e})")
        print("[sheet] no acquisitions tab found in the Google Sheet")
    except Exception as e:
        print(f"[sheet] couldn't read the Google Sheet ({type(e).__name__}: {str(e)[:150]})")
    if Path(ACQ_SHEET_XLSX).exists():
        daily = _parse_acq_sheet(pd.read_excel(ACQ_SHEET_XLSX, sheet_name="Acquisitions", header=None))
        print(f"[sheet] using the uploaded {ACQ_SHEET_XLSX} instead: {len(daily)} filled-in days")
        return daily
    return {}

def sheet_monthly(daily, b):
    """One row per month: acquisitions (sum of the filled-in days, NaN if none), days_filled, days_in_month."""
    rows = []
    for m in MONTHS:
        days = [d.strftime("%Y-%m-%d") for d in pd.date_range(m, _month_end(m))]
        filled = [d for d in days if d in daily]
        rows.append({"acquisitions": sum(daily[d][b] for d in filled) if filled else float("nan"),
                     "days_filled": len(filled), "days_in_month": len(days)})
    return pd.DataFrame(rows, index=MONTH_LABELS)

SHEET_DAILY = read_acq_sheet()
if not SHEET_DAILY:
    raise RuntimeError("[sheet] acquisitions sheet unreadable -- give the service account Viewer access to the sheet, "
                       f"or upload {ACQ_SHEET_XLSX} in Colab. (Acquisitions only come from the sheet.)")
ACQUISITIONS = {b: sheet_monthly(SHEET_DAILY, b) for b in ("olj", "ot")}

for _b in ("olj", "ot"):
    print(f"\n{_b.upper()} acquisitions (sheet):\n{ACQUISITIONS[_b].to_string()}")
    _gaps = ACQUISITIONS[_b][ACQUISITIONS[_b]["days_filled"] < ACQUISITIONS[_b]["days_in_month"]]
    for _lab, _r in _gaps.iterrows():
        print(f"  ⚠️  {_lab}: only {_r['days_filled']} of {_r['days_in_month']} days filled in the sheet")

## 3. Matomo — traffic, channel types, countries

The Matomo client, and traffic, channel types and countries (section 5: 3c, section 7: 3d). Matomo's numbers, in Matomo's words:

| In the report | Matomo API |
|---|---|
| Visits | `API.get` → `nb_visits` |
| Pageviews | `API.get` → `nb_pageviews` |
| Unique visitors | `API.get` → `nb_uniq_visitors` |
| Avg. visit duration | `API.get` → `avg_time_on_site` |
| Pageviews per visit | pageviews ÷ visits (not "actions per visit", which also counts downloads and outlinks) |
| Sources (section 4) | `Referrers.getReferrerType` — Matomo's channel types: Direct Entry, Search Engines, Campaigns, Social Networks, Websites, AI Assistants |
| Countries (section 6) | `UserCountry.getCountry` |

**Brands** are Matomo's **pre-processed** segments on `dimension5` (site 2), in `MATOMO_SEGMENTS` — written **exactly
as Matomo stores them** (e.g. the `:` of the app IDs is `%253A`). Matomo only serves its pre-computed numbers when the
definition matches character for character; any other spelling is computed live, which is slow and can time out
(Cloudflare **524**). Each run checks Matomo's saved segments, prints whether each brand's segment is found and
pre-processed, and uses Matomo's own spelling if it only differs in encoding.

| | OLJ | OT |
|---|---|---|
| Website | `www.lorientlejour.com` | `today.lorientlejour.com` |
| iOS app | `ios:me.whitebeard.olj` | `ios:me.whitebeard.olj.today` |
| Android app | `android:me.whitebeard.olj` | `android:me.whitebeard.olj.today` |

Every call asks for **all rows** (`filter_limit=-1`): Matomo's default of 100 silently dropped small countries in the old
report. Each method is **one call for the six months** — quick when Matomo has archived them. `API.get` asks for **only the 4 metrics used** (unfiltered it collects every plugin's metrics, and anything not archived for the segment is computed on the spot). If the six-month call still times out (typically **the month that just ended, before Matomo's archiver has built it**), the notebook asks **one month at a time** and prints each month's time and Matomo's archive state, so the slow month is obvious; timeouts are retried with a growing pause. The token is sent by POST. Each manual run prints the month's
`dimension5` values with their visits and brand, to check the segments.

In [ ]:
import urllib.parse
import hashlib
import numpy as np

# =====================================================================================================================
# 3. Matomo -- the client + traffic, channel types and countries (social sub-channels: 3c, top articles: 3d)
# =====================================================================================================================
MATOMO_READY = bool(MATOMO_TOKEN and MATOMO_URL and MATOMO_SITE_ID)

# ---- Matomo's words, used everywhere in the report (charts, tables, headings, notes) --------------------------------------
T = dict(engine="Matomo", visits="Visits", visits_lc="visits", visit_lc="visit", users="Unique Visitors",
         users_row="Unique visitors", users_kpi="Unique visitors", users_lc="unique visitors",
         views="Pageviews", views_row="Pageviews", per_visit="Pageviews per Visit",
         per_visit_row="Pageviews / visit", duration="Avg. Visit Duration", duration_row="Avg. visit duration",
         duration_long="Average Visit Duration", sources="Visit Sources")

# ---- Matomo channel types (Referrers.getReferrerType), in Matomo's own words -------------------------------------------
MATOMO_CHANNELS = ["Direct Entry", "Search Engines", "Campaigns", "Social Networks", "Websites", "AI Assistants"]
MATOMO_COLORS = {"Direct Entry": "4472C4", "Search Engines": "26A69A", "Campaigns": "E53935",
                 "Social Networks": "F5C518", "Websites": "8E44AD", "AI Assistants": "F39C12", "Other": "9E9E9E"}

SOURCE_CATS = MATOMO_CHANNELS + ["Other"]     # section 4
SOURCE_COLORS = MATOMO_COLORS

def _monthly_pivot(rows, key, columns):
    """rows -> index = MONTH_LABELS, given columns. Months with no data stay NaN (a gap, not a fake -100%)."""
    if rows.empty:
        return pd.DataFrame(np.nan, index=MONTH_LABELS, columns=columns)
    df = rows.pivot_table(index="month", columns=key, values="sessions", aggfunc="sum", fill_value=0)
    return df.reindex(columns=columns, fill_value=0).reindex(MONTH_LABELS).astype(float)

def source_cats(df):
    """The source categories actually in a section-4 table (its columns minus Total)."""
    return [c for c in df.columns if c != "Total"]

def _decode_all(text):
    """Fully URL-decode (however many times it was encoded) -- to compare two spellings of the same segment."""
    prev = None
    while prev != text:
        prev, text = text, urllib.parse.unquote(text)
    return text

def segment_values(definition):
    """'dimension5==a,dimension5==b' -> ['a', 'b'] (decoded) -- for the dimension5 check."""
    return [_decode_all(part.split("==", 1)[1]) for part in definition.split(",") if "==" in part]

_SEGMENT_RESOLVED = {}
def matomo_segment(brand):
    """The brand's segment definition, spelled EXACTLY as Matomo stores it (so its pre-processed numbers are used).
    The first time, it checks Matomo's saved segments: if one means the same but is spelled differently, that one is
    used; if none matches, it warns that the segment will be computed live (slow)."""
    if brand.name in _SEGMENT_RESOLVED:
        return _SEGMENT_RESOLVED[brand.name]
    mine = MATOMO_SEGMENTS[brand.name]
    chosen = mine
    try:
        saved = _matomo_post({"module": "API", "method": "SegmentEditor.getAll", "idSite": MATOMO_SITE_ID,
                              "format": "JSON", "token_auth": MATOMO_TOKEN})
        same = [sg for sg in (saved or []) if _decode_all(sg.get("definition", "")) == _decode_all(mine)]
        exact = [sg for sg in same if sg.get("definition") == mine]
        hit = (exact or same or [None])[0]
        if hit is None:
            print(f"  ⚠️  [{brand.name}] this segment isn't one of Matomo's saved segments -- Matomo will compute it "
                  "live (slow, can time out). Copy the definition from Matomo into MATOMO_SEGMENTS.")
        else:
            chosen = hit["definition"]
            pre = str(hit.get("auto_archive", "")) in ("1", "true", "True")
            note = "" if hit.get("definition") == mine else " (spelling differed from ⚙️ Settings -- using Matomo's)"
            print(f"  [{brand.name}] Matomo segment '{hit.get('name', '?')}': "
                  + ("pre-processed ✓" if pre else "⚠️ NOT pre-processed -- computed live (slow)") + note)
    except Exception as e:
        print(f"  [{brand.name}] couldn't list Matomo's saved segments ({type(e).__name__}) -- using ⚙️ Settings as is")
    _SEGMENT_RESOLVED[brand.name] = chosen
    return chosen

MATOMO_RETRY_STATUS = {429, 500, 502, 503, 504, 520, 522, 524}   # timeouts / overload -- worth retrying
MATOMO_MAX_TRIES = 6          # first try + 5 retries
MATOMO_FIRST_WAIT = 20        # seconds before the first retry; doubles each time (max 120 s)

def _matomo_post(data, max_tries=None, timeout=150):
    """POST one Matomo API call, retrying timeouts and server errors.

    A 524 is Cloudflare (in front of Matomo) giving up after ~100 s while Matomo is still computing a segment's
    numbers on the fly (shouldn't happen with pre-processed segments -- kept just in case). Matomo usually finishes and
    keeps the result, so the same call a little later is fast."""
    import time
    wait = MATOMO_FIRST_WAIT
    label = f"{data['method']} {data.get('date', '')}"
    tries = max_tries or MATOMO_MAX_TRIES
    for attempt in range(1, tries + 1):
        try:
            resp = requests.post(MATOMO_URL.rstrip("/") + "/index.php", data=data, timeout=timeout)
            if resp.status_code in MATOMO_RETRY_STATUS:
                raise requests.HTTPError(f"{resp.status_code} from {MATOMO_URL}", response=resp)
            resp.raise_for_status()
            out = resp.json()
            if isinstance(out, dict) and out.get("result") == "error":
                raise RuntimeError(f"[Matomo] {data['method']}: {out.get('message')}")   # a real error -- no retry
            return out
        except (requests.Timeout, requests.ConnectionError, requests.HTTPError) as e:
            code = getattr(getattr(e, "response", None), "status_code", None)
            if isinstance(e, requests.HTTPError) and code not in MATOMO_RETRY_STATUS:
                raise                                                                    # 401 / 403 / 404 ... -- no retry
            if attempt == tries:
                if max_tries:                       # the caller has a plan B (one month at a time) -- hand back
                    raise TimeoutError(f"{label}: {code or type(e).__name__} after {tries} tries") from e
                raise RuntimeError(
                    f"[Matomo] {label}: still failing after {tries} tries ({code or type(e).__name__}).\n"
                    "  Matomo is too slow computing the brand segments on the fly (Cloudflare cuts it at ~100 s).\n"
                    "  Try again in a few minutes (Matomo keeps what it already computed), or ask the Matomo admin to\n"
                    "  make the two segments PRE-PROCESSED (Segment editor > 'processed ... pre-processed'), so the\n"
                    "  numbers are ready before the report asks for them.") from e
            print(f"  [Matomo] {label}: {code or type(e).__name__} -- Matomo still computing, retry "
                  f"{attempt}/{tries - 1} in {wait}s")
            time.sleep(wait)
            wait = min(wait * 2, 120)

def _archive_state(result):
    """Matomo's archive state for one period (fetch_archive_state=1), when it reports one."""
    if isinstance(result, dict):
        for k, v in result.items():
            if "archive" in str(k).lower() and not isinstance(v, (dict, list)):
                return str(v)
    return None

def matomo_api(method, segment=None, period="month", date=None, **params):
    """One Matomo Reporting API call (token sent by POST, never in the URL).

    With a date: that call, as is. With no date: the 6 months in ONE call ({'2026-03': ..., ...}) -- quick when Matomo
    has archived them. If that call keeps timing out (a month not archived yet, e.g. the month that just ended), it
    asks ONE MONTH AT A TIME instead, so the archived months come back at once and only the slow one waits; it prints
    each month's time and Matomo's archive state, to show which month Matomo hadn't pre-processed."""
    import time
    base = {"module": "API", "method": method, "idSite": MATOMO_SITE_ID, "period": period, "format": "JSON",
            "filter_limit": "-1",                 # ALL rows -- Matomo's default of 100 silently drops small countries
            "language": "en", "token_auth": MATOMO_TOKEN, **params}
    if segment:
        base["segment"] = segment
    if date is not None:
        return _matomo_post({**base, "date": date})
    rng = f"{MONTHS[0]:%Y-%m-%d},{_month_end(REPORT_MONTH):%Y-%m-%d}"
    try:
        return _matomo_post({**base, "date": rng}, max_tries=2)
    except TimeoutError:
        print(f"  [Matomo] {method}: the 6-month call is too slow -- asking one month at a time to find the slow one")
    out = {}
    for m in MONTHS:
        t0 = time.time()
        res = _matomo_post({**base, "date": f"{m:%Y-%m-%d}", "fetch_archive_state": "1"})
        state = _archive_state(res)
        print(f"     {m:%b %Y}: {time.time() - t0:5.1f}s" + (f"  (archive: {state})" if state else ""))
        out[f"{m:%Y-%m}"] = res
    return out

def _by_month(payload):
    """Matomo's {'2026-03': ..., ...} (keys may carry a day) -> {'2026-03': ...} for our MONTH_LABELS."""
    return {str(k)[:7]: v for k, v in (payload or {}).items()} if isinstance(payload, dict) else {}

def matomo_monthly_traffic(brand, seed=0):
    """page_views, sessions (= visits), users (= unique visitors),
    avg_duration (s), views_per_session (= pageviews per visit)."""
    if MATOMO_READY:
        # API.get, limited to the 4 metrics we use (unfiltered, it collects every plugin's metrics -- goals, e-commerce,
        # media... -- and any of them not archived for the segment is computed on the spot, which is slow). Raw numbers.
        allm = _by_month(matomo_api("API.get", matomo_segment(brand), format_metrics="0",
                                    columns="nb_visits,nb_uniq_visitors,nb_pageviews,avg_time_on_site"))
        rows = []
        for lab in MONTH_LABELS:
            v = allm.get(lab) or {}
            rows.append({"page_views": v.get("nb_pageviews", np.nan), "sessions": v.get("nb_visits", np.nan),
                         "users": v.get("nb_uniq_visitors", np.nan), "avg_duration": v.get("avg_time_on_site", np.nan)})
        df = pd.DataFrame(rows, index=MONTH_LABELS).astype(float)
        df.loc[df["sessions"] == 0] = np.nan                 # a month with no visits at all = no data, not zeros
        if df["users"].isna().all() and df["sessions"].notna().any():
            print(f"  ⚠️  [{brand.name}] Matomo returned no unique visitors per month (processing of monthly unique "
                  "visitors may be off for segments)")
    else:
        rng = np.random.default_rng(seed + 50)
        visits = (380e3 if brand.key == "ot" else 2.1e6) * rng.uniform(0.5, 1.0, N_MONTHS)
        df = pd.DataFrame({"sessions": visits, "page_views": visits * rng.uniform(3.3, 5.4, N_MONTHS),
                           "users": visits * rng.uniform(0.5, 0.62, N_MONTHS),
                           "avg_duration": rng.uniform(500, 640, N_MONTHS)}, index=MONTH_LABELS)
    df["views_per_session"] = df["page_views"] / df["sessions"]
    return df

def matomo_monthly_channels(brand, sessions=None, seed=0):
    """Visits per Matomo channel type per month: index = MONTH_LABELS, columns = SOURCE_CATS + Total."""
    if MATOMO_READY:
        pm = _by_month(matomo_api("Referrers.getReferrerType", matomo_segment(brand)))
        recs, unknown = [], set()
        for lab in MONTH_LABELS:
            for r in pm.get(lab) or []:
                label = r.get("label", "")
                cat = label if label in MATOMO_CHANNELS else "Other"
                if cat == "Other":
                    unknown.add(label)
                recs.append({"month": lab, "category": cat, "sessions": int(r.get("nb_visits", 0))})
        if unknown:
            print(f"  [{brand.name}] Matomo channel types not in MATOMO_CHANNELS (counted as Other): {sorted(unknown)}")
        df = _monthly_pivot(pd.DataFrame(recs, columns=["month", "category", "sessions"]), "category", SOURCE_CATS)
        df.loc[[lab for lab in MONTH_LABELS if not pm.get(lab)]] = np.nan
    else:
        rng = np.random.default_rng(seed + 60)
        base = {"Direct Entry": .54, "Search Engines": .28, "Campaigns": .10, "Social Networks": .04, "Websites": .025,
                "AI Assistants": .015, "Other": 0}
        tot = (sessions if sessions is not None else pd.Series(2.4e5, index=MONTH_LABELS)).reindex(MONTH_LABELS)
        rows = []
        for t in tot:
            w = np.array([base[c] for c in SOURCE_CATS]) * rng.uniform(0.75, 1.25, len(SOURCE_CATS))
            rows.append(np.round(t * w / w.sum()))
        df = pd.DataFrame(rows, index=MONTH_LABELS, columns=SOURCE_CATS)
    if "Other" in df and (df["Other"].fillna(0) == 0).all():
        df = df.drop(columns="Other")                     # only show "Other" when Matomo actually has one
    df["Total"] = df[[c for c in df.columns if c != "Total"]].sum(axis=1, min_count=1)
    return df

def matomo_monthly_country_rows(brand, sessions=None, seed=0):
    """Every (month, country, sessions = visits) row -- what 3b (countries) works from.
    Matomo's 'Unknown' country becomes '(not set)' (never ranked)."""
    if not MATOMO_READY:
        return _example_country_rows(sessions, seed + 5, brand.key)
    pm = _by_month(matomo_api("UserCountry.getCountry", matomo_segment(brand)))
    recs = []
    for lab in MONTH_LABELS:
        for r in pm.get(lab) or []:
            # Matomo's unknown-country row, by its label (language=en) or code xx when sent -- never just "no code"
            unknown = str(r.get("label", "")).strip().lower() in ("unknown", "") or str(r.get("code", "")).lower() == "xx"
            name = "(not set)" if unknown else r.get("label")
            recs.append((lab, name, int(r.get("nb_visits", 0))))
    return pd.DataFrame(recs, columns=["month", "country", "sessions"])

def matomo_segment_check():
    """Read-only: the values of the segment dimension in the report month, with visits -- to check / fill in
    MATOMO_SEGMENTS."""
    dim_id = MATOMO_SEGMENT_DIMENSION.replace("dimension", "")
    rows = matomo_api("CustomDimensions.getCustomDimension", period="month", date=f"{REPORT_MONTH:%Y-%m-%d}",
                      idDimension=dim_id)
    print(f"[Matomo] {MATOMO_SEGMENT_DIMENSION} values in {REPORT_MONTH:%B %Y} (site {MATOMO_SITE_ID}):")
    for r in sorted(rows or [], key=lambda r: -int(r.get("nb_visits", 0)))[:25]:
        owner = next((b for b, d in MATOMO_SEGMENTS.items() if r.get("label") in segment_values(d)), "—")
        print(f"   {int(r.get('nb_visits', 0)):>10,}  {r.get('label')}   -> {owner}")

if MATOMO_READY:
    print(f"Matomo ready -- {MATOMO_URL}, site {MATOMO_SITE_ID}")
    if not os.environ.get("GITHUB_ACTIONS"):
        try:
            matomo_segment_check()
        except Exception as e:
            print(f"[Matomo] segment check skipped ({type(e).__name__}: {str(e)[:120]})")
else:
    print("⚠️  [Matomo not configured -- MATOMO_TOKEN missing -- using EXAMPLE data. The email step will refuse to send it.]")

USING_EXAMPLE_DATA = not MATOMO_READY        # the email step refuses to send example numbers

## 3b. Countries (section 6 of the report) — Matomo

Matomo `UserCountry.getCountry` × month, visits (rows from step 3). Per brand:

| Shown as | Which countries |
|---|---|
| **Full color** | the **top 5 of the report month** — tracked across all 6 months |
| **6.3 rank chart** (only when it happens) | countries that were in the top 5 in an **earlier** month but aren't this month: their sessions each month (same style as 6.2, in the country's fixed color) with the rank beside each point (bold = top 5 that month), and a side note (when they were top 5, where they are now). In the main charts they're part of Others |
| **Others** (grey) | everything else, `(not set)` included |

So the main charts never show more than 5 countries: a country that suddenly jumps into the top 5 this month appears
right away, and one that drops out moves to Others and to the 6.3 rank chart. `(not set)` is never ranked.

Every country has **one fixed color, the same in both brands and every month** (`COUNTRY_COLORS`); countries in the 6.3 rank chart keep
their own color. A country that isn't in the list yet gets a color derived from its name (always the same
one), and the run prints it so you can pin it in the list. Others is always grey.

In [ ]:
TOP_N_COUNTRIES = 5
OTHERS = "Others"

# One color per country, forever, for both brands (and the same as the GA4 report). English names. Keep them distinct.
COUNTRY_COLORS = {
    "France":               "1E5AA8",   # blue
    "Lebanon":              "E53935",   # red
    "United States":        "F4B400",   # yellow
    "Canada":               "00897B",   # teal
    "Switzerland":          "8E24AA",   # purple
    "United Arab Emirates": "6D4C41",   # brown
    "Saudi Arabia":         "2E7D32",   # dark green
    "Belgium":              "F57C00",   # orange
    "United Kingdom":       "7E57C2",   # lavender
    "Germany":              "455A64",   # slate
    "Qatar":                "880E4F",   # maroon
    "Kuwait":               "00ACC1",   # cyan
    "Egypt":                "AFB42B",   # olive
    "Iraq":                 "D81B60",   # pink
    "Syria":                "7CB342",   # light green
    "Jordan":               "5C6BC0",   # periwinkle
    "Cyprus":               "FF7043",   # coral
    "Australia":            "039BE5",   # sky blue
    "Turkey":               "6A1B9A",   # deep purple
    "Spain":                "BF360C",   # rust
}
OTHERS_COLOR = "9E9E9E"
_FALLBACK_COLORS = ["3E2723", "1B5E20", "4A148C", "B71C1C", "0D47A1", "E65100", "006064", "827717", "AD1457", "33691E"]
assert len(set(COUNTRY_COLORS.values())) == len(COUNTRY_COLORS), "two countries share a color"

def country_color(name):
    """Always the same color for the same country (md5, not Python's hash(), which changes every run)."""
    if name == OTHERS:
        return OTHERS_COLOR
    if name in COUNTRY_COLORS:
        return COUNTRY_COLORS[name]
    return _FALLBACK_COLORS[int(hashlib.md5(name.encode()).hexdigest(), 16) % len(_FALLBACK_COLORS)]

def _example_country_rows(sessions=None, seed=0, brand_key="olj"):
    """Example data only."""
    weights = ({"France": .38, "Lebanon": .20, "Canada": .06, "United States": .05, "Switzerland": .045,
                "Belgium": .03, "United Arab Emirates": .03, "Germany": .02, "(not set)": .01, "Morocco": .01}
               if brand_key == "olj" else
               {"Lebanon": .30, "United States": .20, "United Kingdom": .07, "United Arab Emirates": .06,
                "Canada": .05, "Saudi Arabia": .03, "France": .03, "Australia": .02, "(not set)": .01, "India": .02})
    rng = np.random.default_rng(seed + 20)
    tot = (sessions if sessions is not None else pd.Series(2.4e6, index=MONTH_LABELS)).reindex(MONTH_LABELS)
    rows = []
    for lab, t in tot.items():
        w = np.array(list(weights.values())) * rng.uniform(0.8, 1.2, len(weights))
        rest = max(1 - w.sum(), 0.05)
        tail = ["Morocco", "Italy", "Spain", "Netherlands", "Sweden", "Brazil", "Egypt", "Qatar", "Kuwait", "Jordan",
                "Iraq", "Cyprus", "Turkey", "Greece", "Portugal", "Senegal", "Ivory Coast", "Mexico", "Japan", "Nigeria"]
        tw = rng.uniform(0.5, 1.5, len(tail)); tw = tw / tw.sum() * rest
        rows += [(lab, c, int(round(t * x))) for c, x in zip(list(weights) + tail, list(w) + list(tw))]
    df = pd.DataFrame(rows, columns=["month", "country", "sessions"])
    spike_country, spike_month = ("Belgium", MONTH_LABELS[2]) if brand_key == "olj" else ("Saudi Arabia", MONTH_LABELS[3])
    hit = (df["country"] == spike_country) & (df["month"] == spike_month)
    df.loc[hit, "sessions"] *= 4                          # example only: one month in the top 5, then gone
    return df


COUNTRY_META = {}    # brand name -> {"top", "report_month", "former": {country: {"ranks": [...], "top_months": [...],
                     #                                                  "now_rank": int, "now_sessions": float}}}

def _monthly_ranks(rows):
    """{month: [every country, biggest first]} -- (not set) is never ranked."""
    known = rows[rows["country"] != "(not set)"]
    out = {}
    for lab in MONTH_LABELS:
        m = known[known["month"] == lab]
        out[lab] = list(m.groupby("country")["sessions"].sum().sort_values(ascending=False).index) if not m.empty else []
    return out

def countries_by_month(rows, n=TOP_N_COUNTRIES, label=""):
    """index = MONTH_LABELS; columns = the report month's top n (biggest first), Others, Total -- never more than n
    countries. Countries that were top n in an EARLIER month only are counted in Others here; their rank history is
    kept in COUNTRY_META[label]["former"] for the 6.3 rank chart."""
    ranks = _monthly_ranks(rows)
    report_month = next((lab for lab in reversed(MONTH_LABELS) if ranks[lab]), MONTH_LABELS[-1])
    top = ranks[report_month][:n]
    window = rows.groupby("country")["sessions"].sum()
    former = {}
    for lab in MONTH_LABELS:
        for c in ranks[lab][:n]:
            if c not in top and c not in former:
                former[c] = None
    by_month = rows.groupby(["country", "month"])["sessions"].sum()
    for c in sorted(former, key=lambda c: -window.get(c, 0)):
        rk = [ranks[lab].index(c) + 1 if c in ranks[lab] else None for lab in MONTH_LABELS]
        former[c] = {"ranks": rk, "top_months": [lab for lab, r in zip(MONTH_LABELS, rk) if r and r <= n],
                     "sessions": [float(by_month.get((c, lab), np.nan)) for lab in MONTH_LABELS],
                     "now_rank": rk[MONTH_LABELS.index(report_month)],
                     "now_sessions": float(by_month.get((c, report_month), 0))}
    former = dict(sorted(former.items(), key=lambda kv: -window.get(kv[0], 0)))
    r = rows.assign(bucket=rows["country"].where(rows["country"].isin(top), OTHERS))
    df = _monthly_pivot(r, "bucket", top + [OTHERS])
    df["Total"] = df[top + [OTHERS]].sum(axis=1, min_count=1)
    COUNTRY_META[label] = {"top": top, "former": former, "report_month": report_month}

    print(f"  [{label}] top {n} in {report_month}: {', '.join(top)}")
    for c, f in former.items():
        print(f"     left the top {n}: {c} (top {n} in {', '.join(f['top_months'])}; now #{f['now_rank']})")
    unpinned = [c for c in top + list(former) if c not in COUNTRY_COLORS]
    if unpinned:
        print(f"  ⚠️  no fixed color yet for {unpinned} -- using #{', #'.join(country_color(c) for c in unpinned)}; "
              "add them to COUNTRY_COLORS to pin it")
    return df

def country_cats(df):
    return [c for c in df.columns if c != "Total"]

## 4. GA4 — sections 5 and 7 only (marked * and "GA4 data" in the report)

Two sections of this Matomo report use **GA4** on purpose, and the report says so everywhere they appear — "(GA4 data)"
in the title and the contents, a *Different data source: GA4* box at the top, a **\*** on their table titles and on the
At a Glance line that uses them, and a *Source* line under each:

- **Section 5, distribution channels** — the channels are defined by **GA4 session sources** (`cms-9` À la une /
  `cms-34` Morning Brief, `cms-46` / `cms-48` WhatsApp, `fb`, `ig`, `t.co`…), so GA4 gives exactly the GA4 report's figures.
- **Section 7, top articles + top 2 sources** — Matomo can't give each article's sources in time (its *Transitions*
  report is never pre-processed), so GA4 gives the views and the sources.

Same login and brand stream filters as `monthly_report_GA4.ipynb`: service account first, OAuth as a fallback. Nothing
else in this report uses GA4.

In [ ]:
SERVICE_ACCOUNT_FILE = "service_account.json"
GA4_PROPERTY_ID = "328439412"
OAUTH_CLIENT_SECRET_FILE = "oauth_client_secret.json"
OAUTH_TOKEN_FILE = "token.json"
GA4_SCOPES = ["https://www.googleapis.com/auth/analytics.readonly"]

def get_ga4_credentials():
    if os.path.exists(SERVICE_ACCOUNT_FILE):
        from google.oauth2 import service_account as ga_service_account
        return ga_service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=GA4_SCOPES)
    if os.path.exists(OAUTH_CLIENT_SECRET_FILE):
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google_auth_oauthlib.flow import InstalledAppFlow
        creds = None
        if os.path.exists(OAUTH_TOKEN_FILE):
            creds = Credentials.from_authorized_user_file(OAUTH_TOKEN_FILE, GA4_SCOPES)
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(OAUTH_CLIENT_SECRET_FILE, GA4_SCOPES)
                creds = flow.run_local_server(port=0)
            with open(OAUTH_TOKEN_FILE, "w") as f:
                f.write(creds.to_json())
        return creds
    return None

GA4_READY = (os.path.exists(SERVICE_ACCOUNT_FILE) or os.path.exists(OAUTH_CLIENT_SECRET_FILE)) \
            and GA4_PROPERTY_ID != "REPLACE_WITH_YOUR_PROPERTY_ID"

olj_stream_filter = None
ot_stream_filter = None

if GA4_READY:
    from google.analytics.data_v1beta import BetaAnalyticsDataClient
    from google.analytics.data_v1beta.types import (RunReportRequest, DateRange, Metric, Dimension, OrderBy,
                                                    FilterExpression, FilterExpressionList, Filter)
    ga4_client = BetaAnalyticsDataClient(credentials=get_ga4_credentials())

    olj_stream_filter = FilterExpression(filter=Filter(
        field_name="streamName",
        string_filter=Filter.StringFilter(match_type=Filter.StringFilter.MatchType.CONTAINS, value="olj",
                                          case_sensitive=False)))
    ot_stream_filter = FilterExpression(or_group=FilterExpressionList(expressions=[
        FilterExpression(filter=Filter(field_name="streamName", string_filter=Filter.StringFilter(
            match_type=Filter.StringFilter.MatchType.BEGINS_WITH, value="OT", case_sensitive=False))),
        FilterExpression(filter=Filter(field_name="streamName", string_filter=Filter.StringFilter(
            match_type=Filter.StringFilter.MatchType.CONTAINS, value="orient today", case_sensitive=False))),
    ]))

    def ga4_report(start, end, metrics, dimensions=None, order_by_metric=None, limit=None,
                   extra_filter=None, stream_filter=None):
        if stream_filter is None:
            raise ValueError("ga4_report needs a brand's stream_filter (see the Brands section)")
        dim_filter = stream_filter if extra_filter is None else FilterExpression(
            and_group=FilterExpressionList(expressions=[stream_filter, extra_filter]))
        rows, offset, cols = [], 0, None
        while True:
            request = RunReportRequest(
                property=f"properties/{GA4_PROPERTY_ID}",
                date_ranges=[DateRange(start_date=start.strftime("%Y-%m-%d"), end_date=end.strftime("%Y-%m-%d"))],
                metrics=[Metric(name=m) for m in metrics],
                dimensions=[Dimension(name=d) for d in (dimensions or [])],
                dimension_filter=dim_filter, limit=limit or 100000, offset=offset)
            if order_by_metric:
                request.order_bys = [OrderBy(metric=OrderBy.MetricOrderBy(metric_name=order_by_metric), desc=True)]
            resp = ga4_client.run_report(request)
            cols = [d.name for d in resp.dimension_headers] + [m.name for m in resp.metric_headers]
            for row in resp.rows:
                rows.append([v.value for v in row.dimension_values] + [v.value for v in row.metric_values])
            offset += len(resp.rows)
            if limit or not resp.rows or offset >= resp.row_count:
                break
        return pd.DataFrame(rows, columns=cols)

    print(f"GA4 ready -- property {GA4_PROPERTY_ID}")
else:
    print("⚠️  [GA4 not connected -- sections 5 (social) and 7 (articles) use EXAMPLE data]")
# Real data needs both: Matomo (everything) and GA4 (sections 5 and 7). Otherwise the email step refuses to send.
USING_EXAMPLE_DATA = not (MATOMO_READY and GA4_READY)

## 4a. GA4 session sources — section 5 only

One GA4 query per brand (`yearMonth` × `sessionSource`, all rows). Used **only** to split social traffic into its
sub-channels in 4b. The brands' social-network lists are the GA4 report's: anything in them that isn't one of the four
channels becomes *Other Social (minor)*.

In [ ]:
import numpy as np
import hashlib

# =====================================================================================================================
# 4a. GA4 session sources -- ONLY for section 5 (social sub-channels)
# =====================================================================================================================
# --- The brands' social lists (same as the GA4 report): what's left after the 4 channels = 'Other Social (minor)' ---
# OLJ
SOCIAL_NETWORKS_OLJ = {
    "m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
    "mobile.facebook.com", "facebook", "l.instagram.com", "ig", "instagram",
    "instagram.com", "later-linkinbio", "linkin.bio", "t.co", "linkedin.com",
    "lnkd.in", "go.bsky.app", "l.threads.com", "twitter", "x.com", "threads",
    "bluesky", "pinterest.com", "tiktok.com", "snapchat", "snapchat.com",
    "web.whatsapp.com", "cms-46", "reddit.com", "fb", "flipboard", "flipboard.com",
}

# OT
SOCIAL_NETWORKS_OT = {
    "m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
    "mobile.facebook.com", "facebook", "l.instagram.com", "ig", "instagram",
    "instagram.com", "later-linkinbio", "linkin.bio", "t.co", "linkedin.com",
    "lnkd.in", "go.bsky.app", "l.threads.com", "twitter", "x.com", "threads",
    "bluesky", "pinterest.com", "tiktok.com", "snapchat", "snapchat.com",
    "web.whatsapp.com", "cms-48", "reddit.com", "old.reddit.com", "out.reddit.com",
    "fb", "flipboard", "flipboard.com",
    # Note: WhatsApp is "cms-48" for OT (it was "cms-46" for OLJ).
}

# ---- One GA4 pull per brand: month x sessionSource, used only by 4b (social sub-channels) -------------------------------
def ga4_monthly_source_rows(stream_filter, sessions=None, seed=0, whatsapp_source="cms-46"):
    """Every (month, source, sessions) row for the 6 months -- no top-N cut, so nothing is lost."""
    if GA4_READY:
        raw = ga4_report(WINDOW_START, WINDOW_END, metrics=["sessions"], dimensions=["yearMonth", "sessionSource"],
                         stream_filter=stream_filter)
        if raw.empty:
            return pd.DataFrame(columns=["month", "source", "sessions"])
        return pd.DataFrame({"month": raw["yearMonth"].str[:4] + "-" + raw["yearMonth"].str[4:],
                             "source": raw["sessionSource"], "sessions": raw["sessions"].astype(int)})
    # Example data: a realistic spread of raw sources, scaled to the example sessions of each month
    weights = {"(direct)": .48, "google": .24, "bing": .015, "news.google.com": .01, "chatgpt.com": .006,
               "perplexity.ai": .002, "facebook.com": .02, "m.facebook.com": .015, "web.facebook.com": .003,
               "l.instagram.com": .008, "instagram": .003, "t.co": .004, "x.com": .002, whatsapp_source: .03,
               "web.whatsapp.com": .003, "cms-9": .02, "cms-34": .02, "linkedin.com": .004, "reddit.com": .002, "newsletter": .05,
               "mailchimp": .03, "morningbrief": .02, "somesite.com": .02, "msn.com": .015}
    rng = np.random.default_rng(seed + 10)
    tot = (sessions if sessions is not None else pd.Series(2.4e6, index=MONTH_LABELS)).reindex(MONTH_LABELS)
    rows = []
    for lab, t in tot.items():
        w = np.array(list(weights.values())) * rng.uniform(0.7, 1.3, len(weights))
        rows += [(lab, s, int(round(t * x / w.sum()))) for s, x in zip(weights, w)]
    return pd.DataFrame(rows, columns=["month", "source", "sessions"])

## 4b. Distribution channels (section 5 of the report) — GA4

Uses **your channel definitions** on the GA4 rows of 4a (no extra query), laid out exactly like the reference
table: one **Social** band, rows in this fixed order:

| Row | OLJ | OT |
|---|---|---|
| **À la une** (OLJ) / **Morning Brief** (OT) | `cms-9` | `cms-34` |
| WhatsApp | `cms-46`, `web.whatsapp.com` | `cms-48`, `web.whatsapp.com` |
| Facebook | the Facebook list (+ anything containing "facebook") | same |
| X / Twitter | `t.co`, `x.com`, `twitter` | same |
| Instagram | the Instagram list (+ anything containing "instagram") | same |
| Other Social (minor) | the rest of the brand's social-network list (4a): LinkedIn, Reddit, TikTok… | same |

The À la une / Morning Brief codes come from the weekly brief's map (`cms-34` = Morning Brief, `cms-9` = À la une).
They're set per brand in `FEATURED_ROW` below if one ever changes. Newsletters are **not** in this table.

Why GA4: the channels are defined by these GA4 session sources, so GA4 gives the GA4 report's figures.

In [ ]:
# ---- Channel definitions (OLJ / OT) -- edit here if a new source shows up --------------------------------------------
CHANNELS = ["Twitter / X", "Instagram", "Facebook", "WhatsApp"]

CHANNEL_COLORS = {
    "Twitter / X": "#000000",
    "Instagram":   "#E1306C",
    "Facebook":    "#1877F2",
    "WhatsApp":    "#25D366",
}

CHANNEL_SOURCES_OLJ = {
    "Twitter / X": {"t.co", "x.com", "twitter"},
    "Instagram":   {"l.instagram.com", "ig", "instagram", "instagram.com",
                     "about.instagram.com", "later-linkinbio", "linkin.bio"},
    "Facebook":    {"m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
                     "mobile.facebook.com", "facebook", "fb", "web.facebook.com",
                     "m.prod.facebook.com", "apps.facebook.com", "fr-fr.facebook.com"},
    "WhatsApp":    {"cms-46", "web.whatsapp.com"},
}

CHANNEL_SOURCES_OT = {
    "Twitter / X": {"t.co", "x.com", "twitter"},
    "Instagram":   {"l.instagram.com", "ig", "instagram", "instagram.com",
                     "about.instagram.com", "later-linkinbio", "linkin.bio"},
    "Facebook":    {"m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
                     "mobile.facebook.com", "facebook", "fb", "web.facebook.com",
                     "m.prod.facebook.com", "apps.facebook.com", "fr-fr.facebook.com"},
    "WhatsApp":    {"cms-48", "web.whatsapp.com"},  # OT-specific: cms-48 (OLJ uses cms-46)
}

def categorise_channel(s, channel_sources):
    s  = str(s).strip()
    sl = s.lower()
    for channel, keywords in channel_sources.items():
        if s in keywords or sl in keywords:
            return channel
    if "facebook" in sl:
        return "Facebook"
    if "instagram" in sl:
        return "Instagram"
    return None

# ---- The row above WhatsApp: À la une for OLJ, Morning Brief for OT (codes from the weekly brief's map) -----------------
FEATURED_ROW = {
    "olj": ("À la une",      {"cms-9"}),    # cms-9  = À la une
    "ot":  ("Morning Brief", {"cms-34"}),   # cms-34 = Morning Brief
}

OTHER_SOCIAL = "Other Social (minor)"
CHANNEL_LABELS = {"Twitter / X": "X / Twitter"}                     # how the table writes it (like the reference)
TABLE_CHANNEL_ORDER = ["WhatsApp", "Facebook", "Twitter / X", "Instagram"]   # fixed order, as in the reference

def social_rows_for(brand_key):
    """The table's rows, top to bottom: featured row, the 4 channels, Other Social (minor)."""
    return [FEATURED_ROW[brand_key][0]] + TABLE_CHANNEL_ORDER + [OTHER_SOCIAL]

def social_channels_by_month(rows, brand_key, channel_sources, social, label=""):
    """index = MONTH_LABELS; columns = social_rows_for(brand_key)."""
    featured_name, featured_sources = FEATURED_ROW[brand_key]
    r = rows.copy()
    sl = r["source"].astype(str).str.strip().str.lower()
    r["channel"] = r["source"].map(lambda s: categorise_channel(s, channel_sources))
    r.loc[sl.isin(featured_sources), "channel"] = featured_name
    r.loc[r["channel"].isna() & sl.isin(social), "channel"] = OTHER_SOCIAL
    df = _monthly_pivot(r.dropna(subset=["channel"]), "channel", social_rows_for(brand_key))

    in_channel = r[r["channel"].isin(CHANNELS + [featured_name])]
    print(f"\u2705 [{label}] total sessions found for these channels ({MONTH_LABELS[0]} \u2192 {MONTH_LABELS[-1]}):")
    for ch in [featured_name] + CHANNELS:
        print(f"   {ch:<14} {int(in_channel.loc[in_channel['channel'] == ch, 'sessions'].sum()):,}")
    return df

## 4c. Top 10 articles (section 7 of the report) — GA4

Why GA4: Matomo can't give each article's sources in time (its Transitions report isn't pre-processed).

Same logic as the weekly brief's top articles, **without the Web / App split**, for the report month:

- Web events write the article ID to `articleid`, app events to `article_id` — both are pulled (filtered to the brand's
  streams) and **summed by ID**, so an article read on web and app counts once with all its views.
- The ranking comes from a lean query (ID only), so nothing is cut off or lumped into GA4's "(other)" row; only real
  6–7 digit IDs count.
- A second query for just those 10 IDs adds the title (the most frequent one for the ID, with the
  " - L'Orient-Le Jour" / " - L'Orient Today" suffix removed) and the **top 2 sources**, each as a share of *that
  article's* views (e.g. `google 54% · (direct) 22%`), rounded to a whole percent.

In [ ]:
import re

# =====================================================================================================================
# 3e. Top 10 articles of the report month -- same logic as the weekly brief (3b), without the Web / App split
# =====================================================================================================================
TOP_N_ARTICLES = 10
ARTICLE_ID_PATTERN = re.compile(r"^\d{6,7}$")     # valid article IDs only -- drops "(not set)" and junk
ARTICLE_ID_DIMS = ["customEvent:articleid", "customEvent:article_id"]   # web and app send the ID in different fields
TITLE_SUFFIX = re.compile(r"\s*[-–—|]\s*L['’]Orient[- ](Le Jour|Today)\s*$", re.I)   # " - L'Orient-Le Jour" etc.

def _top2_sources(rows):
    """rows: [(source, views)] for one article -> two 'source NN%' strings; NN% = share of THIS article's views,
    computed on the exact views and rounded to a whole percent (same rule as every % in the report)."""
    total = sum(v for _, v in rows)
    if total == 0:
        return "", ""
    out = [f"{src} {int(v / total * 100 + 0.5)}%" for src, v in sorted(rows, key=lambda t: t[1], reverse=True)[:2]]
    return (out + ["", ""])[0], (out + ["", ""])[1]

def ga4_top_articles_month(stream_filter, n=TOP_N_ARTICLES, seed=0):
    """The report month's n most-viewed articles: Article, Article ID, Views, Top source 1, Top source 2."""
    start, end = dt.datetime.combine(REPORT_MONTH, dt.time.min), WINDOW_END
    if GA4_READY:
        def _valid(df):
            return df[df["article_id"].astype(str).str.strip().apply(lambda v: bool(ARTICLE_ID_PATTERN.match(v)))]
        # 1) the VIEW COUNTS from a lean query (ID only): fewest rows, so nothing is cut off or lumped into GA4's
        #    "(other)" row. Web and app rows with the same ID are summed.
        lean = pd.concat([ga4_report(start, end, metrics=["screenPageViews"], dimensions=[d],
                                     stream_filter=stream_filter).rename(columns={d: "article_id"})
                          for d in ARTICLE_ID_DIMS], ignore_index=True)
        if lean.empty:
            return pd.DataFrame(columns=["Article", "Article ID", "Views", "Top source 1", "Top source 2"])
        lean = _valid(lean.rename(columns={"screenPageViews": "views"}))
        lean["views"] = lean["views"].astype(int)
        top = lean.groupby("article_id")["views"].sum().sort_values(ascending=False).head(n).reset_index()
        # 2) titles and sources: a detailed query for just those IDs
        from google.analytics.data_v1beta.types import Filter as _F, FilterExpression as _FE
        ids = [str(i) for i in top["article_id"]]
        parts = []
        for d in ARTICLE_ID_DIMS:
            only_top = _FE(filter=_F(field_name=d, in_list_filter=_F.InListFilter(values=ids)))
            parts.append(ga4_report(start, end, metrics=["screenPageViews"], dimensions=[d, "sessionSource", "pageTitle"],
                                    stream_filter=stream_filter, extra_filter=only_top).rename(columns={d: "article_id"}))
        detail = _valid(pd.concat(parts, ignore_index=True).rename(columns={"screenPageViews": "views"}))
        detail["views"] = detail["views"].astype(int)
        detail = detail[detail["article_id"].isin(top["article_id"])]
        # the most frequent title seen for each ID (guards against a stray differently-formatted title)
        titles = detail.groupby("article_id")["pageTitle"].agg(lambda s: s.value_counts().idxmax() if len(s) else "")
        by_src = detail.groupby(["article_id", "sessionSource"])["views"].sum()
        s1, s2 = {}, {}
        for aid in top["article_id"]:
            rows = list(by_src.loc[aid].items()) if aid in by_src.index.get_level_values(0) else []
            s1[aid], s2[aid] = _top2_sources(rows)
        top["Article"] = top["article_id"].map(titles).fillna("").map(lambda t: TITLE_SUFFIX.sub("", str(t)))
        top["Top source 1"], top["Top source 2"] = top["article_id"].map(s1), top["article_id"].map(s2)
        return top.rename(columns={"article_id": "Article ID", "views": "Views"})[
            ["Article", "Article ID", "Views", "Top source 1", "Top source 2"]]
    # Example data
    rng = np.random.default_rng(seed + 30)
    views = sorted((rng.uniform(0.25, 1.0, n) * (95000 if seed == 0 else 21000)).astype(int), reverse=True)
    pairs = [("google 54%", "(direct) 22%"), ("facebook 41%", "google 30%"), ("(direct) 38%", "newsletter 19%"),
             ("google 47%", "instagram 15%"), ("(direct) 33%", "google 28%"), ("google 39%", "bing 12%"),
             ("newsletter 44%", "(direct) 21%"), ("google 36%", "facebook 20%"), ("(direct) 29%", "google 24%"),
             ("google 31%", "(direct) 26%")][:n]
    return pd.DataFrame({"Article": [f"Example article title {i} — a headline long enough to show how titles wrap"
                                     for i in range(1, n + 1)],
                         "Article ID": [f"15481{i:02d}" for i in range(1, n + 1)], "Views": views,
                         "Top source 1": [p[0] for p in pairs], "Top source 2": [p[1] for p in pairs]})

## 5. Brands — the only brand-specific code

Each `Brand` holds what differs between the two reports. `collect_brand_data` pulls the brand's GA4 numbers and
picks its accounts (step 1) and acquisitions (step 2) into one monthly table; `collect_brand_sources` pulls
its sessions per source category (3b, the weekly brief's lists), per social sub-channel (3c, the brand's channel
definitions + À la une / Morning Brief), per country (3d), and the month's top 10 articles (3e).

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Brand:
    key: str                  # "olj" / "ot" -- prefix used by steps 1 and 2
    name: str                 # "OLJ" / "OT" -- subtitle, file name, email subject
    full_name: str            # title on the report and the footer
    stream_filter: object     # GA4 filter selecting this brand's streams (sections 5 and 7 only)
    social_sources: frozenset # GA4 sessionSource values of social networks (4a) -- section 5 only
    channel_sources: dict         # distribution-channel definitions (4b, = the GA4 report's) -- WhatsApp differs per brand
    whatsapp_source: str          # cms-46 (OLJ) / cms-48 (OT) -- shown in the report's notes
    recipients_variable: str  # GitHub variable that overrides SCHEDULED_RECIPIENTS[name]
    seed: int = 0             # example data only

    def file_stem(self):
        return f"{self.name}.{REPORT_MONTH:%B}.{REPORT_MONTH:%Y}.{REPORT_SOURCE}"   # e.g. OLJ.September.2026.Matomo(.docx)

ALL_BRANDS = {
    "OLJ": Brand("olj", "OLJ", "L'Orient-Le Jour", olj_stream_filter, frozenset(SOCIAL_NETWORKS_OLJ),
                 CHANNEL_SOURCES_OLJ, "cms-46", "MONTHLY_RECIPIENTS_OLJ", 0),
    "OT":  Brand("ot", "OT", "L'Orient Today", ot_stream_filter, frozenset(SOCIAL_NETWORKS_OT),
                 CHANNEL_SOURCES_OT, "cms-48", "MONTHLY_RECIPIENTS_OT", 1),
}
BRANDS = [ALL_BRANDS[n] for n in BUILD_BRANDS]

def collect_brand_data(brand):
    """One DataFrame per brand, index = MONTH_LABELS, with every number the report uses."""
    df = matomo_monthly_traffic(brand, seed=brand.seed)                       # Matomo
    df["accounts"] = ACCOUNTS[brand.key]["Total"].reindex(MONTH_LABELS).astype(float)
    acq = ACQUISITIONS[brand.key].reindex(MONTH_LABELS)
    df["acquisitions"] = acq["acquisitions"].astype(float)
    df["acq_days_filled"], df["acq_days_in_month"] = acq["days_filled"], acq["days_in_month"]
    return df

def collect_brand_sources(brand, d):
    """(section 4 channel types -- Matomo, section 5 distribution channels -- GA4, section 6 countries -- Matomo,
    all countries' rows -- Matomo, for the Excel file)."""
    channels = matomo_monthly_channels(brand, sessions=d["sessions"], seed=brand.seed)                # Matomo
    ga4_rows = ga4_monthly_source_rows(brand.stream_filter, sessions=d["sessions"], seed=brand.seed,
                                       whatsapp_source=brand.whatsapp_source)                          # GA4, section 5
    social = social_channels_by_month(ga4_rows, brand.key, brand.channel_sources, brand.social_sources, label=brand.name)
    country_rows = matomo_monthly_country_rows(brand, sessions=d["sessions"], seed=brand.seed)
    return channels, social, countries_by_month(country_rows, label=brand.name), country_rows

BRAND_DATA, BRAND_SOURCES, BRAND_SOCIAL, BRAND_COUNTRIES, BRAND_ARTICLES, BRAND_COUNTRY_ROWS = {}, {}, {}, {}, {}, {}
for b in BRANDS:
    BRAND_DATA[b.name] = collect_brand_data(b)
    print(f"\n[{b.name}]\n" + BRAND_DATA[b.name].round(2).to_string())
    (BRAND_SOURCES[b.name], BRAND_SOCIAL[b.name], BRAND_COUNTRIES[b.name],
     BRAND_COUNTRY_ROWS[b.name]) = collect_brand_sources(b, BRAND_DATA[b.name])
    for _title, _df in (("Matomo visits by channel type", BRAND_SOURCES[b.name]),
                        ("GA4 distribution channels (section 5)", BRAND_SOCIAL[b.name]),
                        ("Matomo visits by country", BRAND_COUNTRIES[b.name])):
        print(f"\n[{b.name}] {_title}\n" + _df.astype("Int64").to_string())
    BRAND_ARTICLES[b.name] = ga4_top_articles_month(b.stream_filter, seed=b.seed)          # GA4, section 7
    print(f"\n[{b.name}] GA4 top {TOP_N_ARTICLES} articles (section 7), {REPORT_MONTH:%B %Y}\n"
          + BRAND_ARTICLES[b.name].drop(columns=["Article"]).to_string(index=False))

## 6. Charts — same look as the current monthly notebook

Three images per brand: **1.1** page views / sessions / users (stacked), **1.2** views per session and avg session
duration, **2** accounts created and acquisitions. Each point shows its value and the % change vs the month before.

Plus two for **4. Evolution of Traffic Sources** (categories and colors from 3b, i.e. the weekly brief's):
the % share of each category per month (stacked to 100%, total sessions on top), and the sessions per category
as lines with a value badge on every point. The same two charts are drawn for **6. Audience by country** (this month's top 5 in
full color, countries that were top 5 in an earlier month faded with a `*`, then Others).

In [ ]:
import matplotlib
if os.environ.get("GITHUB_ACTIONS") == "true":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

CHART_DIR = Path("monthly_charts"); CHART_DIR.mkdir(exist_ok=True)
CHART_DPI = 150

def format_number(v):
    """4.55M / 619K / 2,338 / 3.11 -- same as the reference report."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    v = float(v); sign = "-" if v < 0 else ""; v = abs(v)
    if v >= 1_000_000:
        out = f"{v/1_000_000:.2f}M"
    elif v >= 10_000:
        out = f"{v/1_000:.0f}K"
    elif v == int(v):
        out = f"{int(v):,}"
    else:
        out = f"{v:.2f}"
    return sign + out

def format_seconds_as_duration(seconds):
    if seconds is None or (isinstance(seconds, float) and np.isnan(seconds)):
        return ""
    seconds = int(round(seconds)); h, rem = divmod(seconds, 3600); m, s = divmod(rem, 60)
    return f"{h}h {m}m {s}s" if h else f"{m}m {s}s"

def pct_change(cur, prev):
    if prev is None or cur is None or pd.isna(prev) or pd.isna(cur) or prev == 0:
        return None
    return (cur - prev) / prev * 100

def pct_parts(pct):
    """One rule for every % change in the report: computed on the EXACT numbers (pct_change), then rounded to a
    whole percent for display (half up). Returns (arrow, 'N%', direction); under ±0.5% it's ('', '0%', 'flat')."""
    r = int(abs(pct) + 0.5)
    if r == 0:
        return "", "0%", "flat"
    return ("▲", f"{r}%", "up") if pct > 0 else ("▼", f"{r}%", "down")

def pct_label(pct, signed=False, space=" "):
    """'▲ 5%' (or '▲ +5%' with signed=True); '0%' when flat."""
    arrow, text, way = pct_parts(pct)
    if way == "flat":
        return text
    return f"{arrow}{space}{('+' if way == 'up' else '-') if signed else ''}{text}"

PCT_COLORS = {"up": "#2E7D32", "down": "#C62828", "flat": "#888888"}

def plot_metric(ax, y_raw, labels, title, line_color, fill_color, display_values=None, tick_fmt=format_number, head=1.3):
    """One trend line: filled area, value on each point (alternating above / below), % change arrows."""
    x = np.arange(len(labels))
    y = np.array(y_raw, dtype=float)
    display_values = display_values or [format_number(v) for v in y]
    ax.plot(x, y, color=line_color, linewidth=2, marker="o", markersize=5, zorder=3)
    ax.fill_between(x, np.nan_to_num(y), 0, color=fill_color, alpha=0.6, zorder=1)
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold", color=line_color, pad=14)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    y_top = np.nanmax(y) * head if np.nanmax(np.nan_to_num(y)) > 0 else 1
    ax.set_ylim(bottom=0, top=y_top)
    ax.grid(axis="y", color="#E0E0E0", linewidth=0.7, zorder=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: tick_fmt(v)))
    for i, (xi, yi) in enumerate(zip(x, y)):
        if np.isnan(yi):
            continue
        offset = y_top * 0.06 if i % 2 == 0 else -y_top * 0.09
        ax.annotate(display_values[i], (xi, yi + offset), ha="center", va="bottom" if offset > 0 else "top",
                    fontsize=8.5, fontweight="bold", color=line_color)
    for i in range(1, len(y)):
        pct = pct_change(y[i], y[i - 1])
        if pct is None:
            continue
        ax.annotate(pct_label(pct), (x[i], y[i] + y_top * 0.14), ha="center", va="bottom",
                    fontsize=8, fontweight="bold", color=PCT_COLORS[pct_parts(pct)[2]])

def _save(fig, name):
    path = CHART_DIR / name
    fig.savefig(path, dpi=CHART_DPI, bbox_inches="tight", facecolor="white")
    plt.show() if os.environ.get("GITHUB_ACTIONS") != "true" else None
    plt.close(fig)
    return str(path)

def build_charts(brand, d):
    labels = MONTH_LABELS
    charts = {}

    # 1.1 -- Pageviews / Sessions / Total Users
    fig, axes = plt.subplots(3, 1, figsize=(9, 10.8))
    fig.suptitle("Traffic Performance", fontsize=14, fontweight="bold", color="#1F3864", x=0.02, ha="left", y=1.0)
    for ax, (title, col, lc, fc) in zip(axes, [(f"A | {T['views']}", "page_views", "#2E75B6", "#DCE9F5"),
                                              (f"B | {T['visits']}", "sessions", "#29B6D8", "#DDF1F6"),
                                              (f"C | {T['users']}", "users", "#1F3864", "#E3E8EF")]):
        plot_metric(ax, d[col], labels, title, lc, fc, head=1.25)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    charts["traffic"] = _save(fig, f"{brand.name}_1_traffic.png")

    # 1.2 -- Views per Session / Avg Session Duration
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    plot_metric(axes[0], d["views_per_session"], labels, T["per_visit"], "#2E9BD6", "#DCEEF7",
                display_values=[f"{v:.2f}" if not pd.isna(v) else "" for v in d["views_per_session"]],
                tick_fmt=lambda v: f"{v:.1f}")
    plot_metric(axes[1], d["avg_duration"], labels, T["duration"], "#E07B39", "#FBE9DD",
                display_values=[format_seconds_as_duration(v) for v in d["avg_duration"]],
                tick_fmt=format_seconds_as_duration)
    fig.tight_layout()
    charts["engagement"] = _save(fig, f"{brand.name}_2_engagement.png")

    # 2 -- Accounts Created / Acquisitions
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    plot_metric(axes[0], d["accounts"], labels, "Accounts Created", "#6A9E3B", "#E7F0DD", head=1.35)
    plot_metric(axes[1], d["acquisitions"], labels, f"{brand.name} Acquisitions", "#C0392B", "#F8DEDB", head=1.35)
    fig.tight_layout()
    charts["accounts"] = _save(fig, f"{brand.name}_3_accounts_acquisitions.png")
    return charts

# ---- 4 + 6. Share / evolution charts (shared: traffic sources in 4, countries in 6) ----------------------------------
SOURCE_MONTH_LABELS = [f"{m:%b %Y}" for m in MONTHS]           # "Aug 2026", like the reference charts

def chart_share(df, cats, color_of, title, file_name, faded=None, legend_title=None):
    """Stacked 100% bars: each category's share of the month's sessions, the month's total on top.
    faded: categories drawn lighter and hatched, with a * in the legend (countries that left the top 5)."""
    faded = faded or {}
    share = df[cats].div(df["Total"].where(df["Total"] > 0), axis=0) * 100
    x = np.arange(len(MONTHS))
    fig, ax = plt.subplots(figsize=(10, 5.8))
    bottom = np.zeros(len(x))
    for cat in cats:
        vals = share[cat].fillna(0).to_numpy()
        fade = cat in faded
        ax.bar(x, vals, 0.55, bottom=bottom, color="#" + color_of(cat), alpha=0.35 if fade else 1,
               hatch="////" if fade else None, label=f"{cat}$^{{*}}$" if fade else cat,
               edgecolor="#" + color_of(cat) if fade else "white", linewidth=0 if fade else 0.6, zorder=2)
        for xi, b, v in zip(x, bottom, vals):
            if v >= 5:                                                # too thin to label below 5%
                ax.text(xi, b + v / 2, f"{v:.1f}%", ha="center", va="center", fontsize=8.5, fontweight="bold",
                        color="#444444" if fade else "white", zorder=3)
        bottom += vals
    for xi, t in zip(x, df["Total"]):
        if not pd.isna(t) and t > 0:
            ax.text(xi, 102, format_number(t), ha="center", va="bottom", fontsize=9.5, fontweight="bold", clip_on=False)
    ax.set_ylim(0, 100)
    ax.set_xticks(x); ax.set_xticklabels(SOURCE_MONTH_LABELS)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.set_ylabel(f"% of {T['visits_lc']}", fontsize=9)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.08), ncol=3, fontsize=8, frameon=True)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=78)
    fig.tight_layout()
    return _save(fig, file_name)

def chart_evolution(df, cats, color_of, title, file_name, faded=None):
    """One line per category (sessions), each point with its value in a badge of the line's color.
    faded: {category: month indexes where it was top 5} -- dashed, light line and badges, except a full-color
    dot + badge in the months listed (countries that left the top 5)."""
    faded = faded or {}
    x = np.arange(len(MONTHS))
    vals = df[cats]
    ymax = float(np.nanmax(vals.to_numpy())) if vals.notna().any().any() else 1.0
    gap = ymax * 0.06                      # minimum vertical distance between two badges of the same month
    fig, ax = plt.subplots(figsize=(10, 5.8))
    for cat in cats:
        if cat in faded:
            ax.plot(x, vals[cat], color="#" + color_of(cat), linewidth=1.6, linestyle="--", alpha=0.4, marker="o",
                    markersize=3.5, label=f"{cat}$^{{*}}$", zorder=2)
            hi = sorted(faded[cat])
            ax.plot(x[hi], vals[cat].iloc[hi], linestyle="none", marker="o", markersize=6, color="#" + color_of(cat),
                    zorder=3)
        else:
            ax.plot(x, vals[cat], color="#" + color_of(cat), linewidth=2, marker="o", markersize=4.5, label=cat, zorder=3)
    top_badge = 0
    for i in range(len(x)):
        # lowest value first, each badge pushed up so it never sits on the one below (the small series bunch up)
        pts = sorted((vals[c].iloc[i], c) for c in cats if not pd.isna(vals[c].iloc[i]))
        prev = -np.inf
        for v, cat in pts:
            y = max(v + gap * 0.55, prev + gap)
            prev = y
            top_badge = max(top_badge, y)
            light = cat in faded and i not in faded[cat]
            ax.text(x[i], y, format_number(v), ha="center", va="center", fontsize=7, fontweight="bold", color="white",
                    zorder=4, bbox=dict(boxstyle="round,pad=0.25", fc="#" + color_of(cat), ec="none",
                                        alpha=0.4 if light else 1))
    ax.set_ylim(-gap * 0.6, max(ymax, top_badge) + gap * 0.8)
    ax.set_xticks(x); ax.set_xticklabels(SOURCE_MONTH_LABELS)
    ax.set_xlim(-0.3, len(x) - 0.7)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: format_number(v) if v >= 0 else ""))
    ax.set_ylabel(T["visits"], fontsize=9)
    ax.grid(axis="y", color="#E0E0E0", linewidth=0.7, zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02), ncol=3, fontsize=8, frameon=True)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=60)
    fig.tight_layout()
    return _save(fig, file_name)

source_color = lambda cat: SOURCE_COLORS.get(cat, "9E9E9E")

def chart_country_ranks(brand):
    """6.3: the countries that were top 5 in an earlier month but aren't now -- their SESSIONS per month (like 6.2,
    each in its fixed color, value badge on top of each point) with the month's RANK beside each point (bold + ringed
    = in the top 5 that month), and a side note per country."""
    meta = COUNTRY_META[brand.name]
    former, n = meta["former"], TOP_N_COUNTRIES
    import matplotlib.gridspec as gridspec
    fig = plt.figure(figsize=(10, 4.6))
    g = gridspec.GridSpec(1, 2, width_ratios=[3, 1.3], left=0.07, right=0.99, top=0.86, bottom=0.1, wspace=0.1)
    ax, side = fig.add_subplot(g[0]), fig.add_subplot(g[1])
    x = np.arange(len(MONTHS))
    vals = {c: np.array(f["sessions"], dtype=float) for c, f in former.items()}
    ymax = max((np.nanmax(v) for v in vals.values() if np.isfinite(v).any()), default=1.0)
    gap = ymax * 0.09
    top_badge = 0
    for c, f in former.items():
        col = "#" + country_color(c)
        v = vals[c]
        ax.plot(x, v, color=col, linewidth=2.2, marker="o", markersize=5, zorder=3)
        for xi, r in enumerate(f["ranks"]):
            if r and r <= n and not np.isnan(v[xi]):          # ring the months it was in the top 5
                ax.plot(x[xi], v[xi], "o", markersize=10, mfc="white", mec=col, mew=2.2, zorder=4)
    for i in range(len(x)):                                    # rank beside each point, spread apart if they'd touch
        pts = sorted((vals[c][i], c) for c in former if not np.isnan(vals[c][i]) and former[c]["ranks"][i])
        prev = -np.inf
        for v, c in pts:
            y = max(v, prev + ymax * 0.06)
            prev = y
            r = former[c]["ranks"][i]
            ax.text(x[i] + 0.09, y, f"#{r}", ha="left", va="center", fontsize=8.5, color="#" + country_color(c),
                    fontweight="bold" if r <= n else "normal", zorder=5,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9))
    for i in range(len(x)):                                    # session badges on top, pushed apart like 6.2
        pts = sorted((vals[c][i], c) for c in former if not np.isnan(vals[c][i]))
        prev = -np.inf
        for v, c in pts:
            y = max(v + gap * 0.6, prev + gap)
            prev = y; top_badge = max(top_badge, y)
            ax.text(x[i], y, format_number(v), ha="center", va="center", fontsize=7.5, fontweight="bold",
                    color="white", zorder=6, bbox=dict(boxstyle="round,pad=0.25", fc="#" + country_color(c), ec="none"))
    ax.set_ylim(0, max(ymax, top_badge) + gap * 0.8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: format_number(v)))
    ax.set_ylabel(T["visits"], fontsize=9)
    ax.grid(axis="y", color="#E0E0E0", linewidth=0.7, zorder=0)
    ax.set_xticks(x); ax.set_xticklabels(SOURCE_MONTH_LABELS, fontsize=8.5)
    ax.set_xlim(-0.35, len(MONTHS) - 0.45)
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
    ax.set_title(f"Countries that left the top {n} — {T['visits_lc']} by month", fontsize=12, fontweight="bold", loc="left", pad=12)
    # side note
    side.axis("off")
    side.text(0, 1, f"Not in the top {n} in {REPORT_MONTH:%B}", fontsize=10, fontweight="bold", va="top")
    y, step = 0.86, min(0.24, 0.7 / max(len(former), 1))
    for c, f in former.items():
        when = ", ".join(f"#{f['ranks'][MONTH_LABELS.index(m)]} in {dt.datetime.strptime(m, '%Y-%m'):%b}"
                         for m in f["top_months"])
        now = (f"Now #{f['now_rank']} · {format_number(f['now_sessions'])} {T['visits_lc']}" if f["now_rank"]
               else f"No {T['visits_lc']} this month")
        side.text(0, y, "●", color="#" + country_color(c), fontsize=12, va="top")
        side.text(0.08, y, c, fontsize=9.5, fontweight="bold", va="top")
        side.text(0.08, y - 0.075, f"Top {n}: {when}\n{now}", fontsize=8.3, color="#555555", va="top", linespacing=1.45)
        y -= step
    side.text(0, 0.0, f"#N beside a point = rank that month\n(bold, ringed = in the top {n}).\n"
                      "Counted in Others in 6.1 and 6.2.", fontsize=8, color="#888888", style="italic", va="bottom",
              linespacing=1.4)
    return _save(fig, f"{brand.name}_8_country_ranks.png")

CHARTS = {}
for b in BRANDS:
    CHARTS[b.name] = build_charts(b, BRAND_DATA[b.name])
    src, ctry = BRAND_SOURCES[b.name], BRAND_COUNTRIES[b.name]
    CHARTS[b.name]["source_share"] = chart_share(src, source_cats(src), source_color, f"{T['sources']} % Share by Month",
                                                 f"{b.name}_4_source_share.png")
    CHARTS[b.name]["source_evolution"] = chart_evolution(src, source_cats(src), source_color, f"{T['sources']} Evolution by Month",
                                                         f"{b.name}_5_source_evolution.png")
    CHARTS[b.name]["country_share"] = chart_share(ctry, country_cats(ctry), country_color,
                                                  f"{T['visits']} by Country — % Share by Month", f"{b.name}_6_country_share.png")
    CHARTS[b.name]["country_evolution"] = chart_evolution(ctry, country_cats(ctry), country_color,
                                                          f"{T['visits']} by Country — Evolution by Month",
                                                          f"{b.name}_7_country_evolution.png")
    CHARTS[b.name]["country_ranks"] = chart_country_ranks(b) if COUNTRY_META[b.name]["former"] else None

## 7. Word — one report per brand

Laid out like a professional advisory report:

| Page | What's on it |
|---|---|
| Cover | minimal and centered: the logo (or the brand name in the headline font), a short rule, "Monthly Performance Report", the month, and *Direction Numérique · Confidential* at the foot. Nothing else (no header / footer) |
| Contents | every section and sub-section with its page number (click to jump) + a *How to read this report* box |
| **At a glance** | the one page for busy readers, titled simply *At a Glance*: the five key figures with ▲/▼, three takeaways (top source, top social channel, top country) and one sessions chart |
| **Section 01** Traffic Performance | 1.1 pageviews / sessions / users · 1.2 views per session / avg duration (+ optional disclaimer) |
| **Section 02** Net Account Creations & Acquisitions | accounts created and acquisitions |
| **Section 03** Table Summary | every headline metric, month over month (a real Word table) |
| **Section 04** Evolution of Traffic Sources | 4.1 share by source · 4.2 sessions by source + month-over-month table |
| **Section 05** Traffic from Social Media & Newsletters **(GA4 data)** | 5.1 distribution channels — GA4 sessions, *Different data source* box, marked * |
| **Section 06** Audience Breakdown by Country | 6.1 share by country · 6.2 sessions by country + month-over-month table |
| **Section 07** Top 10 Articles **(GA4 data)** | the month's most-read articles + top 2 sources — GA4, *Different data source* box, marked * |
| **Appendix A** Definitions & Data Sources | what each number means and where it comes from |

Every section starts on a new page with a *SECTION 0X* label, its title, a one-line description and a navy rule;
sub-sections are numbered (4.1, 4.2…). Headings use Word's Heading 1 / 2 styles, so Word's navigation pane shows the
outline too. Footnote markers are superscript <sup>*</sup>. Running header (brand logo · month) and "Confidential · Page N"
footer on every page except the cover. US Letter, 0.75" margins, Arial.

Page numbers on the contents page are set when the report is built (each section starts its own page). If a page
ever overflows (e.g. a very long disclaimer), select all in Word and press F9 to refresh them.

File name: **`Brand.Month.Year.Matomo.docx`**, e.g. `OLJ.August.2026.Matomo.docx`.

**How every number is explained:** a short *Source:* line under every chart and table (Matomo / CMS / sheet / GA4 and how it's counted); GA4 numbers carry a **\*** ("How to read" box + appendix explain it).

In [ ]:
from docx import Document
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK, WD_TAB_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

NAVY, NAVY_DARK, GREY, TEXT = "1F4D78", "1F3864", "888888", "333333"
UP, UP_BG, DOWN, DOWN_BG = "2E7D32", "EAF6EC", "C62828", "FBEAEA"
CONTENT_W_IN = 8.5 - 2 * 0.75
DOWNLOAD_IN_COLAB = True

SUMMARY_ROWS = [                     # (label, column, how to show the value)
    (T["visits"], "sessions", format_number),
    (T["views_row"], "page_views", format_number),
    (T["users_row"], "users", format_number),
    (T["duration_row"], "avg_duration", format_seconds_as_duration),
    (T["per_visit_row"], "views_per_session", lambda v: f"{v:.2f}"),
    ("Accounts created", "accounts", format_number),
    ("Acquisitions", "acquisitions", format_number),
]

def _el(tag, **attrs):
    e = OxmlElement(tag)
    for k, v in attrs.items():
        e.set(qn(f"w:{k}"), str(v))
    return e

def _run(p, text, size=10, bold=False, color="000000", italic=False):
    r = p.add_run(str(text))
    r.font.size, r.bold, r.italic = Pt(size), bold, italic
    r.font.color.rgb = RGBColor.from_string(color)
    return r

def _spacing(p, before=0, after=0):
    p.paragraph_format.space_before, p.paragraph_format.space_after = Pt(before), Pt(after)

def _border(p, edge, size, color=NAVY, space=4):
    """Paragraph rule (edge = 'top' / 'bottom'); size in 1/8 pt."""
    pPr = p._p.get_or_add_pPr()
    bdr = pPr.find(qn("w:pBdr"))
    if bdr is None:                              # pBdr must sit after pStyle / keepNext / ... (Word's schema order)
        bdr = _el("w:pBdr")
        after = {qn(f"w:{t}") for t in ("shd", "tabs", "suppressAutoHyphens", "kinsoku", "wordWrap", "overflowPunct",
                 "topLinePunct", "autoSpaceDE", "autoSpaceDN", "bidi", "adjustRightInd", "snapToGrid", "spacing",
                 "ind", "contextualSpacing", "mirrorIndents", "suppressOverlap", "jc", "textDirection",
                 "textAlignment", "textboxTightWrap", "outlineLvl", "divId", "cnfStyle", "rPr", "sectPr", "pPrChange")}
        nxt = next((ch for ch in pPr if ch.tag in after), None)
        nxt.addprevious(bdr) if nxt is not None else pPr.append(bdr)
    bdr.append(_el(f"w:{edge}", val="single", sz=size, space=space, color=color))

def _page_break(doc):
    doc.add_paragraph().add_run().add_break(WD_BREAK.PAGE)

def _heading(doc, number, title, subtitle=None, before=14):
    """'1.1 Traffic Trends - Pageviews / ...' with the navy rule underneath, like the reference."""
    p = doc.add_paragraph(); _spacing(p, before, 8)
    _run(p, f"{number} {title}", 14, True)
    if subtitle:
        _run(p, f" - {subtitle}", 12, True)
    _border(p, "bottom", 8)
    p.paragraph_format.keep_with_next = True

def _picture(doc, path, width_in):
    p = doc.add_paragraph(); _spacing(p, 4, 4)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(path, width=Inches(width_in))

def _note(doc, text):
    p = doc.add_paragraph(); _spacing(p, 0, 6)
    _run(p, text, 8.5, color=GREY, italic=True)

def _footer(section, brand):
    ftr = section.footer
    p = ftr.paragraphs[0] if ftr.paragraphs else ftr.add_paragraph()
    _border(p, "top", 8, space=6)
    for pos in (3.25, 6.5):                     # clear the Footer style's default center / right tabs
        p.paragraph_format.tab_stops.add_tab_stop(Inches(pos), WD_TAB_ALIGNMENT.CLEAR)
    p.paragraph_format.tab_stops.add_tab_stop(Inches(CONTENT_W_IN), WD_TAB_ALIGNMENT.RIGHT)
    _run(p, f"{brand.full_name} — Confidential", 8, color=GREY)
    _run(p, "\tPage ", 8, color=GREY)
    r = _run(p, "", 10)
    for el in (_el("w:fldChar", fldCharType="begin"), None, _el("w:fldChar", fldCharType="end")):
        if el is None:
            it = OxmlElement("w:instrText"); it.set(qn("xml:space"), "preserve"); it.text = " PAGE "
            r._r.append(it)
        else:
            r._r.append(el)

def _cell_text(cell, text, size, bold, color, align=WD_ALIGN_PARAGRAPH.CENTER):
    p = cell.paragraphs[0]; _spacing(p, 0, 0); p.alignment = align
    return _run(p, text, size, bold, color)

def _delta_pill(cell, pct):
    """Second line in a table cell: '▲ 19.3%' in green / red on a light tint -- like the reference table."""
    p = cell.add_paragraph(); _spacing(p, 2, 0); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    way = pct_parts(pct)[2]
    r = _run(p, f" {pct_label(pct)} ", 7.5, True, {"up": UP, "down": DOWN, "flat": GREY}[way])
    r._r.get_or_add_rPr().append(_el("w:shd", val="clear", color="auto",
                                     fill={"up": UP_BG, "down": DOWN_BG, "flat": "F2F2F2"}[way]))

def _cell_borders(cell, **edges):
    tcPr = cell._tc.get_or_add_tcPr()
    b = _el("w:tcBorders")
    for edge in ("top", "left", "bottom", "right"):
        if edge in edges:
            sz, color = edges[edge]
            b.append(_el(f"w:{edge}", val="single", sz=sz, space=0, color=color))
        else:
            b.append(_el(f"w:{edge}", val="nil"))
    tcPr.append(b)

def summary_table(doc, d):
    label_w = 1.6
    month_w = (CONTENT_W_IN - label_w) / len(MONTH_LABELS)
    t = doc.add_table(rows=1 + len(SUMMARY_ROWS), cols=1 + len(MONTH_LABELS))
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = False
    widths = [label_w] + [month_w] * len(MONTH_LABELS)
    for i, w in enumerate(widths):
        t.columns[i].width = Inches(w)          # the table grid (what Word / LibreOffice lay out from)
        for c in t.columns[i].cells:
            c.width = Inches(w)
    # header
    _cell_text(t.cell(0, 0), "Metric", 9, True, "555555", WD_ALIGN_PARAGRAPH.LEFT)
    for j, lab in enumerate(MONTH_LABELS, start=1):
        _cell_text(t.cell(0, j), lab, 9, True, "555555")
    for j in range(len(widths)):
        _cell_borders(t.cell(0, j), bottom=(12, "DDDDDD"))
    # rows
    for i, (label, col, fmt) in enumerate(SUMMARY_ROWS, start=1):
        _cell_text(t.cell(i, 0), label, 9, True, TEXT, WD_ALIGN_PARAGRAPH.LEFT)
        vals = d[col].tolist()
        for j, v in enumerate(vals, start=1):
            cell = t.cell(i, j)
            _cell_text(cell, "—" if pd.isna(v) else fmt(v), 9.5, True, NAVY_DARK)
            pct = pct_change(v, vals[j - 2]) if j > 1 else None
            if pct is not None:
                _delta_pill(cell, pct)
        for j in range(len(widths)):
            _cell_borders(t.cell(i, j), bottom=(4, "E6E6E6"))
    for row in t.rows:
        row.height = Inches(0.46)
        for c in row.cells:
            c.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
    return t

# ---- 4 + 6. Month-over-month table (traffic sources, countries) ------------------------------------------------------
SRC_TOTAL_BG, SRC_STRIPE = "0F1B33", "F5F7FA"

def _shade(cell, fill):
    cell._tc.get_or_add_tcPr().append(_el("w:shd", val="clear", color="auto", fill=fill))

def _delta_line(cell, pct, on_dark=False):
    """Second line in a cell: '▲+12.3%' / '▼-6.1%' in green / red (brighter on the navy TOTAL row)."""
    p = cell.add_paragraph(); _spacing(p, 1, 0); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    way = pct_parts(pct)[2]
    color = ({"up": "7CD992", "down": "FF7B7B", "flat": "BBBBBB"} if on_dark else {"up": UP, "down": DOWN, "flat": GREY})[way]
    _run(p, pct_label(pct, signed=True, space=""), 7.5, True, color)

def _set_widths(t, widths):
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = False
    for i, w in enumerate(widths):
        t.columns[i].width = Inches(w)
        for c in t.columns[i].cells:
            c.width = Inches(w)

def mom_table(doc, df, cats, color_of, first_col="Source", faded=None):
    """One row per category + TOTAL; each month's sessions with the % change vs the month before underneath.
    faded: categories shown in grey italics with a * (countries that left the top 5); their top-5 months stay bold."""
    faded = faded or {}
    label_w = 1.75
    month_w = (CONTENT_W_IN - label_w) / len(MONTHS)
    rows = cats + ["Total"]
    t = doc.add_table(rows=1 + len(rows), cols=1 + len(MONTHS))
    widths = [label_w] + [month_w] * len(MONTHS)
    _set_widths(t, widths)
    _cell_text(t.cell(0, 0), first_col, 9, True, "111111", WD_ALIGN_PARAGRAPH.LEFT)
    for j, lab in enumerate(SOURCE_MONTH_LABELS, start=1):
        _cell_text(t.cell(0, j), lab, 9, True, "111111")
    for j in range(len(widths)):
        _cell_borders(t.cell(0, j), bottom=(16, "111111"))
    for i, cat in enumerate(rows, start=1):
        is_total = cat == "Total"
        first = t.cell(i, 0).paragraphs[0]; _spacing(first, 0, 0)
        if is_total:
            _run(first, "TOTAL", 9.5, True, "FFFFFF")
        elif cat in faded:
            _run(first, "● ", 11, False, color_of(cat))
            _run(first, cat, 9, False, GREY, italic=True)
            _sup(first, "*", 9)
        else:
            _run(first, "● ", 11, False, color_of(cat))
            _run(first, cat, 9, False, TEXT)
        vals = df[cat].tolist()
        for j, v in enumerate(vals, start=1):
            cell = t.cell(i, j)
            if cat in faded:                       # grey, except the months it was in the top 5 (bold, normal color)
                was_top = (j - 1) in faded[cat]
                r = _cell_text(cell, "—" if pd.isna(v) else format_number(v), 9, was_top, TEXT if was_top else GREY)
                r.italic = not was_top
            else:
                _cell_text(cell, "—" if pd.isna(v) else format_number(v), 9.5 if is_total else 9,
                           is_total, "FFFFFF" if is_total else TEXT)
            pct = pct_change(v, vals[j - 2]) if j > 1 else None
            if pct is not None:
                _delta_line(cell, pct, on_dark=is_total)
        for j in range(len(widths)):
            c = t.cell(i, j)
            _cell_borders(c, bottom=(4, "E6E6E6"))      # borders before shading (Word's required order)
            if is_total:
                _shade(c, SRC_TOTAL_BG)
            elif i % 2 == 0:
                _shade(c, SRC_STRIPE)
    for row in t.rows:
        row.height = Inches(0.42)
        for c in row.cells:
            c.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
    return t

# ---- 5. Social sub-channels -- period-over-period (totals, with a Δ% column between each pair of months) ---------------
SUB_HEAD_BG, SOCIAL_BAND = "1F3864", "9C27B0"   # the band uses the section-4 Social Networks color

def _sub_header_cell(cell, lines, align=WD_ALIGN_PARAGRAPH.CENTER):
    p = cell.paragraphs[0]; _spacing(p, 0, 0); p.alignment = align
    for k, line in enumerate(lines):
        r = _run(p, line, 6.5, k == 0, "FFFFFF" if k == 0 else "C9D3E6")
        if k < len(lines) - 1:
            r.add_break()
    _shade(cell, SUB_HEAD_BG)

def _tight_margins(t, twips=35):
    """Narrow left/right cell padding (Word's default 0.08" would wrap a 12-column table)."""
    tblPr = t._tbl.tblPr
    mar = _el("w:tblCellMar")
    for edge in ("left", "right"):
        mar.append(_el(f"w:{edge}", w=twips, type="dxa"))
    look = tblPr.find(qn("w:tblLook"))
    look.addprevious(mar) if look is not None else tblPr.append(mar)

def social_table(doc, soc, brand_key):
    """Exactly like the reference: a Social band, then the featured row (À la une / Morning Brief), WhatsApp,
    Facebook, X / Twitter, Instagram, Other Social (minor) -- fixed order, a Δ% column between each pair of months."""
    n = len(MONTHS)
    label_w, total_w = 1.15, 0.55
    delta_w = (CONTENT_W_IN - label_w - n * total_w) / (n - 1)
    widths = [label_w] + sum(([total_w, delta_w] for _ in range(n - 1)), []) + [total_w]
    rows = social_rows_for(brand_key)
    t = doc.add_table(rows=2 + len(rows), cols=len(widths))
    _set_widths(t, widths)
    _tight_margins(t)
    short = [f"{m:%b}" for m in MONTHS]
    _sub_header_cell(t.cell(0, 0), ["Channel"], WD_ALIGN_PARAGRAPH.LEFT)
    for k, m in enumerate(MONTHS):
        _sub_header_cell(t.cell(0, 2 * k + 1), [f"{m:%b %Y}", "total"])
        if k < n - 1:
            _sub_header_cell(t.cell(0, 2 * k + 2), [f"{short[k]}→{short[k + 1]}", "Δ%"])
    band = t.cell(1, 0).merge(t.cell(1, len(widths) - 1))
    p = band.paragraphs[0]; _spacing(p, 0, 0)
    _run(p, "Social", 8, True, "FFFFFF")
    _shade(band, SOCIAL_BAND)
    t.rows[1].height = Inches(0.26)
    for r, key in enumerate(rows):
        i = r + 2
        minor = key == OTHER_SOCIAL
        first = t.cell(i, 0).paragraphs[0]; _spacing(first, 0, 0)
        _run(first, CHANNEL_LABELS.get(key, key), 7.5, not minor, GREY if minor else "111111")
        vals = soc[key].tolist()
        for k, v in enumerate(vals):
            _cell_text(t.cell(i, 2 * k + 1), "—" if pd.isna(v) else f"{int(v):,}", 7, False, GREY if minor else TEXT)
            if k < n - 1:
                c = t.cell(i, 2 * k + 2)
                pct = pct_change(vals[k + 1], v)
                if pct is None:
                    _cell_text(c, "—", 7, False, GREY)
                else:
                    way = pct_parts(pct)[2]
                    _cell_text(c, pct_label(pct, signed=True, space="\u00a0"), 6.3, True,   # no-break space
                               {"up": UP, "down": DOWN, "flat": GREY}[way])
                    _shade(c, {"up": UP_BG, "down": DOWN_BG, "flat": "F2F2F2"}[way])
        if r % 2 == 1:
            for j in [0] + list(range(1, len(widths), 2)):   # stripe the label + total columns (Δ% cells are tinted)
                _shade(t.cell(i, j), SRC_STRIPE)
        t.rows[i].height = Inches(0.32)
    for row in t.rows:
        for c in row.cells:
            c.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
    return t

def _table_title(doc, title, subtitle=None):
    p = doc.add_paragraph(); _spacing(p, 8, 0 if subtitle else 6); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _run(p, title, 11, True, NAVY_DARK)
    p.paragraph_format.keep_with_next = True
    if subtitle:
        p = doc.add_paragraph(); _spacing(p, 0, 6); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        _run(p, subtitle, 8, color=GREY)
        p.paragraph_format.keep_with_next = True

# =====================================================================================================================
# Report layout: cover · contents · one opener page per section · numbered sub-sections · appendix
# =====================================================================================================================
from docx.enum.text import WD_TAB_LEADER

ACCENT_2 = "29B6D8"            # cyan accent bar on the cover (the sessions color of chart 1.1)
INTRO_GREY = "5A6A7A"
RULE_GREY = "BFC9D6"

_RPR_AFTER_SPACING = ("w:w", "w:kern", "w:position", "w:sz", "w:szCs", "w:highlight", "w:u", "w:effect", "w:bdr",
                      "w:shd", "w:fitText", "w:vertAlign", "w:rtl", "w:cs", "w:em", "w:lang", "w:eastAsianLayout")

def _letter_spacing(run, twentieths_pt):
    """Character spacing, inserted where Word's schema expects it (Word rejects out-of-order run properties)."""
    rPr = run._r.get_or_add_rPr()
    el = _el("w:spacing", val=twentieths_pt)
    nxt = next((ch for ch in rPr if ch.tag in {qn(t) for t in _RPR_AFTER_SPACING}), None)
    nxt.addprevious(el) if nxt is not None else rPr.append(el)

def _sup(p, text="*", size=9, color=GREY, bold=False):
    r = _run(p, text, size, bold, color)
    r.font.superscript = True
    return r

def _setup_styles(doc):
    """Heading 1 / 2 in Arial navy, so Word's navigation pane shows the report's outline."""
    for name, size in (("Heading 1", 22), ("Heading 2", 12.5)):
        st = doc.styles[name]
        rf = st.element.get_or_add_rPr().get_or_add_rFonts()
        for a in ("asciiTheme", "hAnsiTheme", "eastAsiaTheme", "cstheme"):
            rf.attrib.pop(qn(f"w:{a}"), None)
        rf.set(qn("w:ascii"), "Arial"); rf.set(qn("w:hAnsi"), "Arial")
        st.font.size, st.font.bold = Pt(size), True
        st.font.color.rgb = RGBColor.from_string(NAVY_DARK)

_BOOKMARK_ID = [0]
def _bookmark(p, name):
    """Start a bookmark in p; call the returned function after adding the runs to close it."""
    _BOOKMARK_ID[0] += 1
    i = str(_BOOKMARK_ID[0])
    p._p.get_or_add_pPr()
    p._p.append(_el("w:bookmarkStart", id=i, name=name))
    return lambda: p._p.append(_el("w:bookmarkEnd", id=i))

def _field(p, instr, cached, size, bold, color):
    """A Word field (e.g. PAGEREF) showing `cached` until Word refreshes it."""
    runs = []
    for kind in ("begin", "instr", "separate", "value", "end"):
        if kind == "value":
            runs.append(_run(p, str(cached), size, bold, color)); continue
        r = p.add_run(); r.font.size = Pt(size)
        if kind == "instr":
            it = OxmlElement("w:instrText"); it.set(qn("xml:space"), "preserve"); it.text = instr; r._r.append(it)
        else:
            r._r.append(_el("w:fldChar", fldCharType=kind))
        runs.append(r)
    return runs

def _header(section, brand):
    hdr = section.header
    p = hdr.paragraphs[0] if hdr.paragraphs else hdr.add_paragraph()
    for pos in (3.25, 6.5):
        p.paragraph_format.tab_stops.add_tab_stop(Inches(pos), WD_TAB_ALIGNMENT.CLEAR)
    p.paragraph_format.tab_stops.add_tab_stop(Inches(CONTENT_W_IN), WD_TAB_ALIGNMENT.RIGHT)
    logo = logo_path(brand)
    try:
        if not logo:
            raise FileNotFoundError
        p.add_run().add_picture(logo, height=Inches(0.17))
    except Exception:
        _serif(_run(p, brand.full_name, 9, True, "111111"))
    _run(p, f"\t{REPORT_NAME}  ·  {REPORT_MONTH:%B %Y}", 7.5, color=GREY)
    _border(p, "bottom", 4, color=RULE_GREY, space=4)

class _Report:
    """Tracks the page each section / sub-section starts on (every section starts a page), for the contents page."""
    def __init__(self, doc, accent=NAVY):
        self.doc, self.page, self.toc, self.accent = doc, 1, [], accent

    def new_page(self):
        """Start a new page. The break is attached to the end of the last paragraph when there is one, so a page that
        is filled exactly can't push an empty break paragraph onto a blank page of its own."""
        from docx.text.paragraph import Paragraph
        body = self.doc.element.body
        last = [el for el in body if el.tag != qn("w:sectPr")][-1:]
        if last and last[0].tag == qn("w:p"):
            Paragraph(last[0], self.doc._body).add_run().add_break(WD_BREAK.PAGE)
        else:
            _page_break(self.doc)
        self.page += 1

    def section(self, num, title, intro):
        doc = self.doc
        self.new_page()
        bm = f"sec_{num}"
        p = doc.add_paragraph(); _spacing(p, 0, 4); p.paragraph_format.keep_with_next = True
        _border(p, "top", 24, color="111111", space=6)
        _letter_spacing(_run(p, "APPENDIX" if not str(num).isdigit() else f"SECTION {int(num):02d}", 8.5, True,
                             self.accent), 40)
        p = doc.add_paragraph(style="Heading 1"); _spacing(p, 0, 6); p.paragraph_format.keep_with_next = True
        close = _bookmark(p, bm)
        _serif(_run(p, title if str(num).isdigit() else f"{num}. {title}", 26, True, "111111"))
        close()
        p = doc.add_paragraph(); _spacing(p, 0, 14)
        _serif(_run(p, intro, 11, color="444444", italic=True))
        _border(p, "bottom", 6, color="111111", space=8)
        self.toc.append((1, str(num), title, bm, self.page))

    def sub(self, num, title, new_page=False):
        doc = self.doc
        if new_page:
            self.new_page()
        bm = "sec_" + num.replace(".", "_")
        p = doc.add_paragraph(style="Heading 2"); _spacing(p, 8, 8); p.paragraph_format.keep_with_next = True
        close = _bookmark(p, bm)
        _serif(_run(p, num, 13, True, self.accent))
        _serif(_run(p, "   " + title, 13, True, "111111"))
        close()
        _border(p, "bottom", 4, color=RULE_GREY, space=4)
        self.toc.append((2, num, title, bm, self.page))

# ---- Newspaper look: serif headlines, brand accent, front-page cover --------------------------------------------------
SERIF = HEADLINE_FONT or "Georgia"

def _serif(run):
    run.font.name = SERIF
    run.element.get_or_add_rPr().get_or_add_rFonts().set(qn("w:eastAsia"), SERIF)
    return run

def _accent(brand):
    return BRAND_ACCENT.get(brand.name, "B3151B")

def logo_path(brand):
    """The brand's logo as a local image file, or None: LOGO_FILES first, then LOGO_URLS (downloaded once)."""
    local = (LOGO_FILES.get(brand.name) or "").strip()
    if local and Path(local).exists():
        return local
    url = (LOGO_URLS.get(brand.name) or "").strip()
    if not url:
        embedded = globals().get("LOGO_EMBEDDED", {}).get(brand.name)
        if not embedded:
            return None
        import base64
        out = f"logo_{brand.name}_embedded.png"
        Path(out).write_bytes(base64.b64decode(embedded))
        return out
    try:
        import requests as _rq
        resp = _rq.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        ctype = resp.headers.get("content-type", "").split(";")[0].strip().lower()
        ext = {"image/png": ".png", "image/jpeg": ".jpg", "image/jpg": ".jpg", "image/gif": ".gif"}.get(ctype)
        if ext is None:
            print(f"[{brand.name}] logo link isn't a PNG/JPG/GIF ({ctype or 'unknown type'}) -- Word can't embed it "
                  "(SVG included); using the text masthead instead")
            return None
        out = f"logo_{brand.name}{ext}"
        Path(out).write_bytes(resp.content)
        return out
    except Exception as e:
        print(f"[{brand.name}] couldn't download the logo ({type(e).__name__}: {str(e)[:100]}) -- using the text masthead")
        return None

def _logo_size(path, max_w=4.0, max_h=0.85):
    try:
        from PIL import Image as _Img
        with _Img.open(path) as im:
            w, h = im.size
        return min(max_w, max_h * w / h)
    except Exception:
        return 3.0

_CAT_PHRASE = {"Direct Entry": "Direct entries", "Campaigns": "Campaigns", "Websites": "Other websites",
               "Direct": "Direct visits", "Search Engines": "Search engines", "Social Networks": "Social networks",
               "AI Assistants": "AI assistants", "Internal/Newsletters": "Newsletters and internal links",
               "Other": "Other referrers"}

def cover_story(brand, d, src, social, countries):
    """(headline, [takeaway sentences]) written from the month's numbers -- for the At a glance page."""
    month, prev_m = f"{REPORT_MONTH:%B}", (f"{PREV_MONTH:%B}" if PREV_MONTH else "")
    cur, prev = d["sessions"].iloc[-1], d["sessions"].iloc[-2] if len(d) > 1 else None
    pct = pct_change(cur, prev)
    if pct is None or pd.isna(cur):
        headline = f"{brand.full_name} in {month}: the numbers"
    else:
        verb = ("surge" if pct >= 10 else "climb" if pct >= 2 else "drop" if pct <= -10 else "slip" if pct <= -2 else None)
        headline = (f"Sessions {verb} {pct_parts(pct)[1]} to {format_number(cur)} in {month}" if verb
                    else f"Sessions hold steady at {format_number(cur)} in {month}")
    out = []
    last = src.iloc[-1]
    if not pd.isna(last["Total"]) and last["Total"] > 0:
        cats = source_cats(src)
        cat = max(cats[1:], key=lambda c: 0 if pd.isna(last[c]) else last[c])      # the biggest after direct
        out.append(f"{_CAT_PHRASE.get(cat, cat)} bring {last[cat] / last['Total'] * 100:.0f}% of {T['visits_lc']}, "
                   f"the largest source after {_CAT_PHRASE.get(cats[0], cats[0]).lower()}.")
    # social: the biggest of section 5's channels -- GA4 sessions, so the sentence ends with the GA4 "*"
    s_last = social.iloc[-1][[c for c in CHANNELS if c in social.columns]]
    if s_last.notna().any() and s_last.max() > 0:
        ch = s_last.idxmax()
        p = pct_change(s_last[ch], social.iloc[-2][ch]) if len(social) > 1 else None
        out.append(f"{CHANNEL_LABELS.get(ch, ch)} leads social with {format_number(s_last[ch])} GA4 sessions"
                   + (f" ({pct_label(p, signed=True, space='').lstrip('▲▼')} vs {prev_m})." if p is not None else ".")
                   + "*")
    ccats = [c for c in country_cats(countries) if c != OTHERS]
    row = countries.iloc[-1]
    if ccats and row[ccats].notna().any() and row["Total"] > 0:
        top_now = row[ccats].idxmax()
        prev_row = countries.iloc[-2][ccats] if len(countries) > 1 else None
        top_prev = prev_row.idxmax() if prev_row is not None and prev_row.notna().any() else top_now
        share = row[top_now] / row["Total"] * 100
        out.append(f"{top_now} remains the largest audience ({share:.0f}% of {T['visits_lc']})." if top_now == top_prev
                   else f"{top_now} overtakes {top_prev} as the largest audience ({share:.0f}% of {T['visits_lc']}).")
    return headline, out

def cover_chart(brand, d):
    """Sessions over the six months, the report month in the brand accent (At a glance page)."""
    accent = "#" + _accent(brand)
    vals = d["sessions"].to_numpy(dtype=float)
    x = np.arange(len(vals))
    fig, ax = plt.subplots(figsize=(10, 2.6))
    ax.bar(x, np.nan_to_num(vals), 0.6, color=["#C9CED6"] * (len(vals) - 1) + [accent], zorder=2)
    for xi, v in zip(x, vals):
        if not np.isnan(v):
            ax.text(xi, v, format_number(v), ha="center", va="bottom", fontsize=11, fontweight="bold",
                    color=accent if xi == x[-1] else "#333333", family="serif")
    ax.set_xticks(x); ax.set_xticklabels([f"{m:%b}".upper() for m in MONTHS], fontsize=9, color="#555555")
    ax.set_ylim(0, np.nanmax(vals) * 1.25 if np.nanmax(np.nan_to_num(vals)) > 0 else 1)
    ax.set_yticks([])
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#111111")
    fig.tight_layout()
    path = CHART_DIR / f"{brand.name}_0_glance.png"
    fig.savefig(path, dpi=CHART_DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return str(path)

COVER_KPIS = [(T["visits"], "sessions"), (T["views"], "page_views"), (T["users_kpi"], "users"),
              ("New accounts", "accounts"), ("Acquisitions", "acquisitions")]

def _cover(doc, brand):
    """Minimal, centered, newspaper feel: the logo, a thick-and-thin rule around the report name, the month in
    large serif, and a dateline-style line at the foot. Nothing else."""
    ink = "1B1B1B"
    p = doc.paragraphs[0] if doc.paragraphs else doc.add_paragraph()
    _spacing(p, 175, 20); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    logo = logo_path(brand)
    try:
        if not logo:
            raise FileNotFoundError
        p.add_run().add_picture(logo, width=Inches(_logo_size(logo, max_w=4.6, max_h=0.95)))
    except Exception:
        _serif(_run(p, brand.full_name, 42, True, ink))
    # the newspaper rule: thick over thin, with the report name between
    p = doc.add_paragraph(); _spacing(p, 0, 26); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _border(p, "top", 24, color=ink, space=5)
    _border(p, "bottom", 6, color=ink, space=5)
    _letter_spacing(_run(p, REPORT_NAME.upper(), 9, True, ink), 80)
    p = doc.add_paragraph(); _spacing(p, 0, 0); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _serif(_run(p, f"{REPORT_MONTH:%B %Y}", 34, False, ink))
    p = doc.add_paragraph(); _spacing(p, 8, 0); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _serif(_run(p, f"{MONTHS[0]:%B} – {REPORT_MONTH:%B %Y}", 10, False, GREY, italic=True))
    if USING_EXAMPLE_DATA:
        p = doc.add_paragraph(); _spacing(p, 16, 0); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        _run(p, "EXAMPLE DATA — DO NOT SEND", 8, True, DOWN)
    # foot: a thin rule and a dateline
    p = doc.add_paragraph(); _spacing(p, 250 if not USING_EXAMPLE_DATA else 226, 0)
    for pos in (3.25, 6.5):
        p.paragraph_format.tab_stops.add_tab_stop(Inches(pos), WD_TAB_ALIGNMENT.CLEAR)
    p.paragraph_format.tab_stops.add_tab_stop(Inches(CONTENT_W_IN / 2), WD_TAB_ALIGNMENT.CENTER)
    p.paragraph_format.tab_stops.add_tab_stop(Inches(CONTENT_W_IN), WD_TAB_ALIGNMENT.RIGHT)
    _border(p, "top", 4, color=ink, space=6)
    _letter_spacing(_run(p, "DIRECTION NUMÉRIQUE", 7, False, GREY), 40)
    _letter_spacing(_run(p, f"\tBEIRUT  ·  VOL. {REPORT_MONTH.year}  ·  NO. {REPORT_MONTH.month:02d}", 7, False, GREY), 40)
    _letter_spacing(_run(p, "\tCONFIDENTIAL", 7, False, GREY), 40)

def _at_a_glance(doc, rep, brand, d, src, social, countries):
    """The one page a busy reader needs: key figures, three takeaways, one chart -- titled simply 'At a Glance'."""
    accent = _accent(brand)
    _, takeaways = cover_story(brand, d, src, social, countries)
    rep.new_page()
    bm = "sec_glance"
    p = doc.add_paragraph(); _spacing(p, 0, 4)
    _border(p, "top", 24, color="111111", space=6)
    _letter_spacing(_run(p, f"{brand.name}  ·  {REPORT_MONTH:%B %Y}".upper(), 8.5, True, accent), 40)
    p = doc.add_paragraph(style="Heading 1"); _spacing(p, 0, 6)
    close = _bookmark(p, bm)
    _serif(_run(p, "At a Glance", 26, True, "111111"))
    close()
    _border(p, "bottom", 6, color="111111", space=8)
    _spacing(doc.add_paragraph(), 0, 6)
    rep.toc.append((1, "", "At a Glance", bm, rep.page))
    # key figures
    t = doc.add_table(rows=1, cols=len(COVER_KPIS)); _set_widths(t, [CONTENT_W_IN / len(COVER_KPIS)] * len(COVER_KPIS))
    _tight_margins(t, 90)
    for j, (label, col) in enumerate(COVER_KPIS):
        cell = t.cell(0, j)
        cur = d[col].iloc[-1]
        pct = pct_change(cur, d[col].iloc[-2]) if len(d) > 1 else None
        _cell_borders(cell, top=(18, "111111"), bottom=(6, "111111"), **({"left": (4, "BBBBBB")} if j else {}))
        p = cell.paragraphs[0]; _spacing(p, 6, 0)
        _letter_spacing(_run(p, label.upper(), 7, True, GREY), 20)
        p = cell.add_paragraph(); _spacing(p, 2, 0)
        _serif(_run(p, "—" if pd.isna(cur) else format_number(cur), 20, True, "111111"))
        p = cell.add_paragraph(); _spacing(p, 0, 6)
        if pct is None:
            _run(p, " ", 8)
        else:
            _run(p, pct_label(pct), 8, True, {"up": UP, "down": DOWN, "flat": GREY}[pct_parts(pct)[2]])
            _run(p, f"  vs {PREV_MONTH:%b}", 8, color=GREY)
    _source(doc, f"{T['visits']}, {T['views'].lower()}, {T['users_lc']} — Matomo (brand segment). New accounts — CMS. "
                 "Acquisitions — acquisitions sheet. ▲/▼ = change vs the previous month.")
    # takeaways
    if takeaways:
        p = doc.add_paragraph(); _spacing(p, 22, 8)
        _letter_spacing(_run(p, "KEY TAKEAWAYS", 8, True, accent), 30)
        for k, text in enumerate(takeaways, start=1):
            p = doc.add_paragraph(); _spacing(p, 0, 8)
            p.paragraph_format.left_indent = Inches(0.35)
            p.paragraph_format.first_line_indent = Inches(-0.35)
            p.paragraph_format.tab_stops.add_tab_stop(Inches(0.35))
            _serif(_run(p, f"{k}", 13, True, accent))
            ga4_line = text.endswith("*")
            _serif(_run(p, "\t" + (text[:-1] if ga4_line else text), 12, False, "222222"))
            if ga4_line:
                _sup(p, "*", 11, accent, True)
        if any(t.endswith("*") for t in takeaways):
            p = doc.add_paragraph(); _spacing(p, 0, 4); p.paragraph_format.left_indent = Inches(0.35)
            _sup(p, "*", 8, GREY, True)
            _run(p, " From GA4 (section 05), not Matomo.", 8, color=GREY, italic=True)
    # one chart
    p = doc.add_paragraph(); _spacing(p, 18, 6)
    _letter_spacing(_run(p, f"{T['visits']}, {MONTHS[0]:%B} – {REPORT_MONTH:%B %Y}".upper(), 8, True, accent), 30)
    p = doc.add_paragraph(); _spacing(p, 0, 2); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(cover_chart(brand, d), width=Inches(CONTENT_W_IN))
    _source(doc, f"Matomo, brand segment — {T['visits_lc']} per month. Full detail in Sections 01 (traffic), 04 "
                 "(sources), 05 (distribution channels, GA4*), 06 (countries) and 07 (articles, GA4*).")

def _contents(doc, rep, anchor, brand):
    """Fill the contents page (inserted before `anchor`, once every section's page is known)."""
    for level, num, title, bm, page in rep.toc:
        p = anchor.insert_paragraph_before()
        pf = p.paragraph_format
        tabs = pf.tab_stops
        if level == 1:
            _spacing(p, 12, 3)
            tabs.add_tab_stop(Inches(0.45))
            size, bold, num_color, title_color = 11, True, _accent(brand), "111111"
        else:
            _spacing(p, 2, 2)
            pf.left_indent = Inches(0.45)
            tabs.add_tab_stop(Inches(0.95))
            size, bold, num_color, title_color = 9.5, False, GREY, TEXT
        tabs.add_tab_stop(Inches(CONTENT_W_IN), WD_TAB_ALIGNMENT.RIGHT, WD_TAB_LEADER.DOTS)
        link = _el("w:hyperlink"); link.set(qn("w:anchor"), bm); link.set(qn("w:history"), "1")
        runs = [_run(p, num, size, True, num_color), _run(p, "\t" + title, size, bold, title_color),
                _run(p, "\t", size, False, GREY)] + _field(p, f" PAGEREF {bm} \\h ", page, size, bold, title_color)
        p._p.append(link)
        for r in runs:
            link.append(r._r)
    # reading guide box under the list
    spacer = anchor.insert_paragraph_before(); _spacing(spacer, 18, 0)
    t = doc.add_table(rows=1, cols=1); _set_widths(t, [CONTENT_W_IN])
    cell = t.cell(0, 0); _shade(cell, "F2F5F9")
    p = cell.paragraphs[0]; _spacing(p, 6, 4); p.paragraph_format.left_indent = Inches(0.12)
    _letter_spacing(_run(p, "HOW TO READ THIS REPORT", 8, True, NAVY), 30)
    lines = [("▲ / ▼", "change versus the previous month (green = up, red = down), calculated on the exact figures "
                       "and rounded to the nearest whole percent, so it can differ slightly from a change worked out "
                       "on the rounded numbers shown. 0% = less than ±0.5%."),
             ("Matomo", f"every number comes from Matomo ({brand.name}'s segment), in Matomo's words (visits, "
                        "unique visitors, pageviews) — except:"),
             ("*", "= adapted from GA4, not Matomo: sections 05 (distribution channels) and 07 (top articles), marked "
                   "\"GA4 data\", and any number with a *. A box at the top of those sections says why."),
             ("Source:", "under every chart and table says exactly where its numbers come from."),
             ("Appendix A", "defines every metric and grouping used in this report.")]
    for k, (key, text) in enumerate(lines):
        p = cell.add_paragraph(); _spacing(p, 1, 6 if k == len(lines) - 1 else 1)
        p.paragraph_format.left_indent = Inches(0.12)
        if key == "*":
            _sup(p, "*", 9, NAVY, True)
        else:
            _run(p, key, 8.5, True, NAVY_DARK)
        _run(p, "  " + text, 8.5, color=TEXT)
    anchor._p.addprevious(t._tbl)

def _source_callout(doc, why):
    """A shaded box at the top of a section that uses GA4 instead of Matomo: says so, why, and that the figures
    aren't directly comparable with the Matomo visits in the rest of the report."""
    t = doc.add_table(rows=1, cols=1); _set_widths(t, [CONTENT_W_IN])
    cell = t.cell(0, 0)
    _cell_borders(cell, left=(24, "1B1B1B"))
    _shade(cell, "F2F5F9")
    p = cell.paragraphs[0]; _spacing(p, 5, 2); p.paragraph_format.left_indent = Inches(0.1)
    _letter_spacing(_run(p, "DIFFERENT DATA SOURCE: GA4", 8, True, "1B1B1B"), 30)
    p = cell.add_paragraph(); _spacing(p, 0, 5); p.paragraph_format.left_indent = Inches(0.1)
    _run(p, why + " These are GA4 figures, so they aren't directly comparable with the Matomo visits in the rest "
               "of this report.", 8.5, color=TEXT)
    _spacing(doc.add_paragraph(), 0, 2)

def _source(doc, text, ga4=False):
    """'Source: …' under a chart or table -- how its numbers are obtained. ga4=True adds the * of GA4 data."""
    p = doc.add_paragraph(); _spacing(p, 2, 8)
    _run(p, "Source", 7.5, True, GREY)
    if ga4:
        _sup(p, "*", 7.5, GREY, True)
    _run(p, ": " + text, 7.5, color=GREY, italic=True)

def _note_star(doc, text):
    """Grey italic note starting with a superscript *."""
    p = doc.add_paragraph(); _spacing(p, 0, 6)
    _sup(p, "*", 8.5)
    _run(p, " " + text, 8.5, color=GREY, italic=True)

def _definitions(doc, brand):
    featured_name, featured_codes = FEATURED_ROW[brand.key]
    acq = "Basic + Premium" if brand.key == "olj" else "OT"
    rows = [
    ] + [
        ("Visits", f"Matomo visits (nb_visits) in {brand.name}'s segment of site {MATOMO_SITE_ID}. A visit that viewed "
                   "both sites is counted in each brand's report."),
        ("Pageviews", "Matomo pageviews (nb_pageviews)."),
        ("Unique visitors", "Matomo unique visitors (nb_uniq_visitors), de-duplicated within each month."),
        ("Avg. visit duration", "Matomo average time on site (avg_time_on_site)."),
        ("Pageviews per visit", "Pageviews ÷ visits (not Matomo's 'actions per visit', which also counts downloads "
                                "and outlinks)."),
    ] + [
        ("Accounts created", "All new CMS accounts in the month (web, app and other sign-ups), Beirut time."),
        ("Acquisitions", f"New subscriptions from the acquisitions sheet ({acq}); a month not fully filled in is flagged."),
        ("Traffic sources", "Matomo channel types (Matomo's own classification): Direct Entry, Search Engines, "
                            "Campaigns, Social Networks, Websites, AI Assistants."),
        ("Distribution channels *", f"GA4 sessions (not Matomo), grouped with the GA4 report's definitions: "
                                    f"{featured_name} ({', '.join(sorted(featured_codes))}), WhatsApp ({brand.whatsapp_source}, "
                                    "web.whatsapp.com), Facebook, X / Twitter and Instagram by their referrers and tags; "
                                    "Other Social = the rest of the social-network list."),
        ("Countries", f"Matomo country ({T['visits_lc']}). The charts show the top {TOP_N_COUNTRIES} of the report month; Others = every "
                      f"other country, including unknown. A country that was top {TOP_N_COUNTRIES} in an earlier month "
                      "only appears in 6.3 (its rank by month). Each country keeps the same colour in every report."),
        ("Top articles *", f"GA4 (not Matomo): the {TOP_N_ARTICLES} articles with the most page views in the report "
                           "month, counted by article ID (web articleid + app article_id, only real 6–7 digit IDs). Top "
                           "sources: the two GA4 session sources with the most views of that article, as a share of its views."),
        ("*", "Adapted from GA4, not Matomo. GA4 counts sessions and page views, Matomo visits and pageviews — "
              "the two aren't directly comparable."),
        ("Change (▲ / ▼)", "Percentage change versus the previous month, calculated on the exact (unrounded) figures and "
                           "then rounded to the nearest whole percent. Because the numbers shown are rounded too "
                           "(e.g. 2.78M), recomputing a change from them can give a slightly different result. "
                           "0% (grey) means a change of less than ±0.5%."),
    ]
    t = doc.add_table(rows=1 + len(rows), cols=2); _set_widths(t, [1.8, CONTENT_W_IN - 1.8])
    for j, h in enumerate(("Term", "Definition")):
        c = t.cell(0, j); _cell_text(c, h, 9, True, "FFFFFF", WD_ALIGN_PARAGRAPH.LEFT); _shade(c, NAVY_DARK)
    for i, (term, text) in enumerate(rows, start=1):
        _cell_text(t.cell(i, 0), term, 9, True, NAVY_DARK, WD_ALIGN_PARAGRAPH.LEFT)
        _cell_text(t.cell(i, 1), text, 8.5, False, TEXT, WD_ALIGN_PARAGRAPH.LEFT)
        for j in range(2):
            _cell_borders(t.cell(i, j), bottom=(4, "E6E6E6"))
            if i % 2 == 0:
                _shade(t.cell(i, j), SRC_STRIPE)
    for row in t.rows:
        for c in row.cells:
            c.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            for p in c.paragraphs:
                _spacing(p, 3, 3)

ARTICLE_TITLE_CLIP = 110

def articles_table(doc, arts):
    """# | Article | Views | Top sources -- the month's leaderboard (a real Word table)."""
    widths = [0.5, 4.2, 0.85, CONTENT_W_IN - 0.5 - 4.2 - 0.85]
    t = doc.add_table(rows=1 + len(arts), cols=4)
    _set_widths(t, widths)
    for j, (h, al) in enumerate((("#", WD_ALIGN_PARAGRAPH.LEFT), ("Article", WD_ALIGN_PARAGRAPH.LEFT),
                                 ("Views", WD_ALIGN_PARAGRAPH.RIGHT), ("Top sources", WD_ALIGN_PARAGRAPH.LEFT))):
        _cell_text(t.cell(0, j), h, 9, True, "111111", al)
        _cell_borders(t.cell(0, j), bottom=(16, "111111"))
    for i, (_, a) in enumerate(arts.reset_index(drop=True).iterrows(), start=1):
        _serif(_cell_text(t.cell(i, 0), str(i), 12, True, _accent_ink(), WD_ALIGN_PARAGRAPH.LEFT))
        title = str(a["Article"]) or f"Article {a['Article ID']}"
        title = title if len(title) <= ARTICLE_TITLE_CLIP else title[:ARTICLE_TITLE_CLIP - 1].rstrip() + "…"
        _serif(_cell_text(t.cell(i, 1), title, 9.5, False, "111111", WD_ALIGN_PARAGRAPH.LEFT))
        _spacing(t.cell(i, 1).paragraphs[0], 2, 0)
        p = t.cell(i, 1).add_paragraph(); _spacing(p, 0, 2)
        _run(p, f"ID {a['Article ID']}", 7, False, GREY)
        _cell_text(t.cell(i, 2), f"{int(a['Views']):,}", 9.5, True, "111111", WD_ALIGN_PARAGRAPH.RIGHT)
        c = t.cell(i, 3)
        _cell_text(c, a["Top source 1"] or "—", 8.5, False, TEXT, WD_ALIGN_PARAGRAPH.LEFT)
        if a["Top source 2"]:
            p = c.add_paragraph(); _spacing(p, 1, 0)
            _run(p, a["Top source 2"], 8.5, False, GREY)
        for j in range(4):
            _cell_borders(t.cell(i, j), bottom=(4, "E6E6E6"))
            if i % 2 == 0:
                _shade(t.cell(i, j), SRC_STRIPE)
    for k, row in enumerate(t.rows):
        row.height = Inches(0.28 if k == 0 else 0.4)
        for c in row.cells:
            c.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
    return t

def _accent_ink():
    return "1B1B1B"

def build_monthly_docx(brand, d, charts, path, src, social, countries, articles=None):
    doc = Document()
    sec = doc.sections[0]
    sec.page_width, sec.page_height = Inches(8.5), Inches(11)
    sec.left_margin = sec.right_margin = sec.top_margin = sec.bottom_margin = Inches(0.75)
    sec.different_first_page_header_footer = True            # the cover has no header / footer
    normal = doc.styles["Normal"]
    normal.font.name, normal.font.size = "Arial", Pt(10)
    normal.element.get_or_add_rPr().get_or_add_rFonts().set(qn("w:eastAsia"), "Arial")
    _setup_styles(doc)
    _header(sec, brand)
    _footer(sec, brand)
    rep = _Report(doc, _accent(brand))
    featured_name = FEATURED_ROW[brand.key][0]

    # ---- cover + contents page ----
    _cover(doc, brand)
    rep.new_page()
    p = doc.add_paragraph(); _spacing(p, 0, 4)
    _border(p, "top", 24, color="111111", space=6)
    _letter_spacing(_run(p, f"{brand.name}  ·  {REPORT_MONTH:%B %Y}".upper(), 8.5, True, _accent(brand)), 40)
    p = doc.add_paragraph(); _spacing(p, 0, 6)
    _serif(_run(p, "Contents", 26, True, "111111"))
    _border(p, "bottom", 6, color="111111", space=8)
    contents_anchor = doc.add_paragraph()                  # the contents list goes here once page numbers are known

    # ---- At a glance ----
    _at_a_glance(doc, rep, brand, d, src, social, countries)

    # ---- 1. Traffic performance ----
    rep.section(1, "Traffic Performance",
                f"How much {brand.full_name} was read each month: {T['views'].lower()}, {T['visits_lc']} and "
                f"{T['users_lc']}, and how engaged those {T['visits_lc']} were.")
    rep.sub("1.1", f"{T['views']}, {T['visits']} & {T['users']}")
    _picture(doc, charts["traffic"], 5.3)
    _source(doc, "Matomo, brand segment — A pageviews (nb_pageviews), B visits (nb_visits), C unique visitors "
                 "(nb_uniq_visitors), per month. ▲/▼ = change vs the previous month.")
    rep.sub("1.2", f"{T['per_visit']} & {T['duration_long']}", new_page=True)
    disclaimer = DISCLAIMERS.get(brand.name, "").strip()
    if disclaimer:
        p = doc.add_paragraph(); _spacing(p, 0, 12)
        from docx.enum.text import WD_COLOR_INDEX
        r = _run(p, "Disclaimer:", 10, True, italic=True); r.font.highlight_color = WD_COLOR_INDEX.YELLOW
        _run(p, f" {disclaimer}", 10)
    _picture(doc, charts["engagement"], CONTENT_W_IN)
    _source(doc, "Matomo, brand segment — pageviews per visit = pageviews ÷ visits; average visit duration = "
                 "avg_time_on_site.")

    # ---- 2. Accounts & acquisitions ----
    rep.section(2, "Net Account Creations & Acquisitions",
                "New registered accounts (CMS) and acquisitions — new paying subscriptions, from the acquisitions sheet — per month.")
    _picture(doc, charts["accounts"], CONTENT_W_IN)
    gaps = [f"{lab} ({int(r.acq_days_filled)} of {int(r.acq_days_in_month)} days)"
            for lab, r in d.iterrows() if r.acq_days_filled < r.acq_days_in_month]
    _source(doc, "Accounts created — the CMS (every new account: web, app and other sign-ups). Acquisitions — the "
                 "acquisitions sheet (" + ("Basic + Premium" if brand.key == "olj" else "OT") + ")."
                 + (f" Not yet complete in the sheet: {', '.join(gaps)}." if gaps else ""))

    # ---- 3. Table summary ----
    rep.section(3, "Table Summary", "Every headline metric side by side, with the change versus the previous month.")
    _table_title(doc, "Headline Metrics — Month-over-Month", "Value per month  ·  % change vs. prior month")
    summary_table(doc, d)
    _source(doc, "Visits, pageviews, unique visitors, visit duration, pageviews per visit — Matomo (brand segment). "
                 "Accounts created — CMS. Acquisitions — acquisitions sheet.")

    # ---- 4. Traffic sources ----
    rep.section(4, "Evolution of Traffic Sources",
                f"Where {T['visits_lc']} come from, grouped by Matomo's own channel types.")
    rep.sub("4.1", f"Share of {T['visits']} by Source")
    _picture(doc, charts["source_share"], CONTENT_W_IN)
    _source(doc, "Matomo, brand segment — visits by channel type (Referrers.getReferrerType); each bar = 100% of "
                 "the month's visits, total on top.")
    rep.sub("4.2", f"{T['visits']} by Source & Month-over-Month Change", new_page=True)
    _picture(doc, charts["source_evolution"], 6.2)
    _table_title(doc, f"{T['sources']} — Month-over-Month Deltas")
    mom_table(doc, src, source_cats(src), source_color, "Source")
    _source(doc, "Matomo, brand segment — visits by channel type, Matomo's own classification of each visit's "
                 "referrer. Total = sum of the channel types.")

    # ---- 5. Distribution channels -- GA4 ----
    rep.section(5, "Traffic from Social Media & Newsletters (GA4 data)",
                f"Sessions from each distribution channel — {featured_name}, WhatsApp and the social networks — month by month.")
    _source_callout(doc, f"These channels are defined by GA4 session sources ({', '.join(sorted(FEATURED_ROW[brand.key][1]))} "
                         f"for {featured_name}, {brand.whatsapp_source} for WhatsApp, fb, ig, t.co…), so this section "
                         "uses GA4 — the same figures as the GA4 report. GA4 counts sessions, Matomo counts visits.")
    rep.sub("5.1", "Distribution Channels — Period-over-Period")
    _table_title(doc, "Distribution Channels — Period-over-Period*", "GA4 sessions per period  ·  % change vs. prior period")
    social_table(doc, social, brand.key)
    _source(doc, "GA4, brand streams — sessions by session source, grouped with the GA4 report's channel definitions: "
                 f"{featured_name} = {', '.join(sorted(FEATURED_ROW[brand.key][1]))}; WhatsApp = {brand.whatsapp_source}, "
                 "web.whatsapp.com; Facebook / Instagram / X = their referrers and tags (fb, ig…); Other Social = the "
                 "rest of the social-network list.", ga4=True)

    # ---- 6. Countries ----
    cats = country_cats(countries)
    former = COUNTRY_META.get(brand.name, {}).get("former", {})
    rep.section(6, "Audience Breakdown by Country",
                f"Where readers are: this month's top {TOP_N_COUNTRIES} countries, tracked over six months, against "
                "the rest of the world" + (f"; countries that left the top {TOP_N_COUNTRIES} are in 6.3." if former else "."))
    rep.sub("6.1", f"Share of {T['visits']} by Country")
    _picture(doc, charts["country_share"], CONTENT_W_IN)
    _note(doc, f"Only the top {TOP_N_COUNTRIES} countries of {REPORT_MONTH:%B} are shown in 6.1 and 6.2. "
               + (f"A country that was in the top {TOP_N_COUNTRIES} in an earlier month but not in {REPORT_MONTH:%B} "
                  f"is counted in Others here and shown on its own in 6.3: {', '.join(former)}." if former else
                  f"No other country was in the top {TOP_N_COUNTRIES} during the period, so there is no 6.3 this month."))
    _source(doc, "Matomo, brand segment — visits by country (UserCountry.getCountry, every country). Others = all "
                 "the other countries, unknown included. Every country is in the Excel file sent with this report.")
    rep.sub("6.2", f"{T['visits']} by Country & Month-over-Month Change", new_page=True)
    _picture(doc, charts["country_evolution"], 6.0)
    _table_title(doc, f"{T['visits']} by Country — Month-over-Month Deltas")
    mom_table(doc, countries, cats, country_color, "Country")
    _source(doc, f"Matomo, brand segment — visits by country. Top {TOP_N_COUNTRIES} = the countries with the most visits "
                 f"in {REPORT_MONTH:%B %Y}, tracked over the 6 months"
                 + (f"; Others includes {', '.join(former)}, top {TOP_N_COUNTRIES} earlier (see 6.3)" if former else "")
                 + ". Each country keeps the same colour in every report.")
    if former:
        rep.sub("6.3", f"Countries That Left the Top {TOP_N_COUNTRIES}", new_page=True)
        p = doc.add_paragraph(); _spacing(p, 0, 8)
        _serif(_run(p, f"These countries were in the top {TOP_N_COUNTRIES} in an earlier month but not in "
                       f"{REPORT_MONTH:%B}, so 6.1 and 6.2 count them in Others. {T['visits']} per month as in 6.2, in each "
                       f"country's usual colour; the rank beside each point is its position that month (bold and "
                       f"ringed = in the top {TOP_N_COUNTRIES}).", 10, color="444444", italic=True))
        _picture(doc, charts["country_ranks"], CONTENT_W_IN)
        _source(doc, "Matomo, brand segment — visits by country; rank = position among all countries that month.")

    # ---- 7. Top articles -- GA4 ----
    if articles is not None and len(articles):
        rep.section(7, f"Top {len(articles)} Articles (GA4 data)",
                    f"The {len(articles)} most-read articles of {REPORT_MONTH:%B %Y}, and the two sources that brought "
                    "each one the most views.")
        _source_callout(doc, "Matomo can't give each article's traffic sources in time (its Transitions report isn't "
                             "pre-processed), so this section uses GA4 for both the views and the sources — the same "
                             "figures as the GA4 report.")
        _table_title(doc, f"Most-Read Articles — {REPORT_MONTH:%B %Y}*", "GA4 page views in the month  ·  top 2 sources")
        articles_table(doc, articles)
        _source(doc, "GA4, brand streams — page views summed by article ID (web articleid + app article_id; only real "
                     "6–7 digit IDs). Top sources = the two GA4 session sources with the most views of that article, as "
                     "a share of the article's own views, rounded to a whole percent.", ga4=True)

    # ---- Appendix ----
    rep.section("A", "Definitions & Data Sources", "What each number in this report means and where it comes from.")
    _definitions(doc, brand)

    _contents(doc, rep, contents_anchor, brand)
    cp = doc.core_properties                                   # what Word shows as the document's title
    cp.title = f"{brand.full_name} — {REPORT_NAME} — {REPORT_MONTH:%B %Y}"
    cp.comments = "Generated by monthly_report_matomo.ipynb"
    cp.subject, cp.author = f"{brand.name} {REPORT_NAME}, {MONTHS[0]:%B} – {REPORT_MONTH:%B %Y}", "Direction Numérique"
    cp.keywords = f"{brand.name}; Matomo; monthly report; {REPORT_MONTH:%Y-%m}"
    zoom = doc.settings.element.find(qn("w:zoom"))          # the default template's zoom lacks a required attribute
    if zoom is not None and zoom.get(qn("w:percent")) is None:
        zoom.set(qn("w:percent"), "100")
    doc.save(path)
    return path

EXPORTED_FILES = {}
for b in BRANDS:
    EXPORTED_FILES[b.name] = build_monthly_docx(b, BRAND_DATA[b.name], CHARTS[b.name], b.file_stem() + ".docx",
                                            BRAND_SOURCES[b.name], BRAND_SOCIAL[b.name], BRAND_COUNTRIES[b.name],
                                            BRAND_ARTICLES.get(b.name))
    print(f"[{b.name}] saved {os.path.abspath(EXPORTED_FILES[b.name])}")

if DOWNLOAD_IN_COLAB:
    try:
        from google.colab import files as colab_files
        for f in EXPORTED_FILES.values():
            colab_files.download(f)
    except ImportError:
        pass

## 7b. Excel — every country's visits and share (one file per brand)

`OLJ.September.2026.Matomo.Countries.xlsx`: **every country** Matomo recorded for the brand (not just the report's top 5),
over the six months —

| Sheet | What's in it |
|---|---|
| **Visits by country** | visits per country per month, sorted by the report month, *Unknown* last, **Total** row (`SUM`) |
| **Share of visits** | each country's share of the month's visits — **formulas** on the first sheet (`0.0%`), Total = 100% |
| **About** | source, segment, period, and how the numbers are counted |

Saved next to the Word file and attached to the same email.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

XLSX_FONT = "Arial"
_HEAD_FILL = PatternFill("solid", fgColor="1F3864")
_TOTAL_FILL = PatternFill("solid", fgColor="E3E8EF")
_THIN = Side(style="thin", color="D9D9D9")

def _countries_grid(rows):
    """rows (month, country, sessions=visits) -> DataFrame: one row per country, one column per month (visits).
    Sorted by the report month's visits; Matomo's unknown country ("(not set)") last, named "Unknown"."""
    if rows is None or rows.empty:
        return pd.DataFrame(columns=MONTH_LABELS)
    g = rows.pivot_table(index="country", columns="month", values="sessions", aggfunc="sum")
    g = g.reindex(columns=MONTH_LABELS)
    have = [m for m in MONTH_LABELS if m in set(rows["month"])]
    g[have] = g[have].fillna(0)                      # a country with no visits in a month that has data = 0
    last = MONTH_LABELS[-1] if MONTH_LABELS[-1] in g else g.columns[-1]
    known = g.drop(index=[i for i in g.index if i == "(not set)"]).sort_values(last, ascending=False)
    unknown = g.loc[[i for i in g.index if i == "(not set)"]].rename(index={"(not set)": "Unknown"})
    return pd.concat([known, unknown])

def _write_grid(ws, grid, title, subtitle, value_cell):
    """Title rows, a header row, one row per country, a TOTAL row. value_cell(r, c) -> what goes in each cell."""
    ws["A1"] = title; ws["A1"].font = Font(name=XLSX_FONT, size=13, bold=True, color="1F3864")
    ws["A2"] = subtitle; ws["A2"].font = Font(name=XLSX_FONT, size=9, italic=True, color="666666")
    header_row, first = 4, 5
    labels = [f"{dt.datetime.strptime(m, '%Y-%m'):%b %Y}" for m in grid.columns]
    for c, text in enumerate(["Country"] + labels, start=1):
        cell = ws.cell(row=header_row, column=c, value=text)
        cell.font = Font(name=XLSX_FONT, bold=True, color="FFFFFF"); cell.fill = _HEAD_FILL
        cell.alignment = Alignment(horizontal="left" if c == 1 else "center", vertical="center")
    total_row = first + len(grid)
    for i, country in enumerate(grid.index):
        r = first + i
        ws.cell(row=r, column=1, value=country).font = Font(name=XLSX_FONT, color="888888" if country == "Unknown" else "111111")
        for j, m in enumerate(grid.columns, start=2):
            cell = ws.cell(row=r, column=j, value=value_cell(r, j, grid.iloc[i, j - 2], total_row))
            cell.font = Font(name=XLSX_FONT)
            cell.border = Border(bottom=_THIN)
        ws.cell(row=r, column=1).border = Border(bottom=_THIN)
    ws.cell(row=total_row, column=1, value="Total").font = Font(name=XLSX_FONT, bold=True)
    ws.cell(row=total_row, column=1).fill = _TOTAL_FILL
    for j in range(2, len(grid.columns) + 2):
        col = get_column_letter(j)
        cell = ws.cell(row=total_row, column=j, value=f"=SUM({col}{first}:{col}{total_row - 1})" if len(grid) else 0)
        cell.font = Font(name=XLSX_FONT, bold=True); cell.fill = _TOTAL_FILL
    ws.column_dimensions["A"].width = 34
    for j in range(2, len(grid.columns) + 2):
        ws.column_dimensions[get_column_letter(j)].width = 13
    ws.freeze_panes = ws.cell(row=first, column=2)
    return first, total_row

def export_countries_xlsx(brand, rows, path):
    """The brand's countries workbook: every country Matomo has, visits per month, and each month's share (formulas)."""
    grid = _countries_grid(rows)
    span = f"{MONTHS[0]:%B %Y} – {REPORT_MONTH:%B %Y}"
    wb = Workbook()
    ws = wb.active; ws.title = "Visits by country"
    first, total_row = _write_grid(
        ws, grid, f"{brand.full_name} — Visits by country (Matomo)",
        f"{span} · every country Matomo recorded for the brand's segment · Unknown = country not detected",
        lambda r, c, v, tr: None if pd.isna(v) else int(v))
    for row in ws.iter_rows(min_row=first, max_row=total_row, min_col=2, max_col=len(grid.columns) + 1):
        for cell in row:
            cell.number_format = "#,##0"
    # shares: formulas pointing at the visits sheet, so they follow any change there
    ws2 = wb.create_sheet("Share of visits")
    src = "'Visits by country'"
    _write_grid(
        ws2, grid, f"{brand.full_name} — Share of visits by country (Matomo)",
        f"{span} · each country's visits ÷ all visits of the month (the Total row of 'Visits by country')",
        lambda r, c, v, tr: (None if pd.isna(v) else
                             f"=IFERROR({src}!{get_column_letter(c)}{r}/{src}!{get_column_letter(c)}${tr},0)"))
    for row in ws2.iter_rows(min_row=first, max_row=total_row, min_col=2, max_col=len(grid.columns) + 1):
        for cell in row:
            cell.number_format = "0.0%"
    for j in range(2, len(grid.columns) + 2):         # the Total row of the shares sheet = 100%
        col = get_column_letter(j)
        ws2.cell(row=total_row, column=j, value=f"=SUM({col}{first}:{col}{total_row - 1})")
    # about
    ws3 = wb.create_sheet("About")
    notes = [
        ("Source", f"Matomo ({MATOMO_URL}), site {MATOMO_SITE_ID}, method UserCountry.getCountry, all rows (filter_limit=-1)."),
        ("Brand", f"{brand.full_name} — Matomo segment: {MATOMO_SEGMENTS[brand.name]}"),
        ("Period", f"{span}, one column per month."),
        ("Visits", "A visit is counted for the brand if it viewed at least one of the brand's pages; a visit that viewed "
                   "both sites is counted in each brand's file."),
        ("Share of visits", "Each country's visits divided by the month's total visits (all countries, Unknown included). "
                            "The cells are formulas on the 'Visits by country' sheet."),
        ("Unknown", "Visits whose country Matomo couldn't detect."),
        ("Report", f"The Word report shows the top {TOP_N_COUNTRIES} countries of {REPORT_MONTH:%B %Y} + Others; this file "
                   "has every country."),
        ("Generated", f"{dt.datetime.now():%d %B %Y, %H:%M} by monthly_report_matomo.ipynb"),
    ]
    ws3["A1"] = "About this file"; ws3["A1"].font = Font(name=XLSX_FONT, size=13, bold=True, color="1F3864")
    for i, (k, v) in enumerate(notes, start=3):
        ws3.cell(row=i, column=1, value=k).font = Font(name=XLSX_FONT, bold=True)
        c = ws3.cell(row=i, column=2, value=v); c.font = Font(name=XLSX_FONT); c.alignment = Alignment(wrap_text=True, vertical="top")
    ws3.column_dimensions["A"].width = 18; ws3.column_dimensions["B"].width = 110
    wb.calculation.fullCalcOnLoad = True               # Excel computes the share formulas when the file is opened
    wb.save(path)
    return path

EXPORTED_XLSX = {}
for b in BRANDS:
    EXPORTED_XLSX[b.name] = export_countries_xlsx(b, BRAND_COUNTRY_ROWS.get(b.name), b.file_stem() + ".Countries.xlsx")
    n = BRAND_COUNTRY_ROWS[b.name]["country"].nunique() if b.name in BRAND_COUNTRY_ROWS else 0
    print(f"[{b.name}] saved {Path(EXPORTED_XLSX[b.name]).resolve()}  ({n} countries)")
    if os.environ.get("GITHUB_ACTIONS") != "true":
        try:
            from google.colab import files
            files.download(EXPORTED_XLSX[b.name])
        except Exception:
            pass

## 8. Email — one per brand, Word + Excel

**Who gets it:** test runs (Colab, or a manual *Run workflow*) go only to `TEST_RECIPIENTS`; the scheduled run on the
1st goes to each brand's `SCHEDULED_RECIPIENTS`, or to the GitHub variable `MONTHLY_RECIPIENTS_OLJ` /
`MONTHLY_RECIPIENTS_OT` when set. By hand it previews each email and asks **y/N** before sending.

Gmail uses the same `GMAIL_ADDRESS` + `GMAIL_APP_PASSWORD` (App Password) as the weekly brief.
It refuses to send if the numbers are example data (Matomo not connected).

**Subject:** `Brand.Month.Year.Matomo`, e.g. `OLJ.August.2026.Matomo` — the same name as the attached Word file.

In [ ]:
import getpass
import smtplib
from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

IS_SCHEDULED = os.environ.get("GITHUB_EVENT_NAME") == "schedule"
IN_CI = os.environ.get("GITHUB_ACTIONS") == "true"

def _split_emails(text):
    return [e.strip() for e in str(text).replace(";", ",").replace("\n", ",").split(",") if e.strip()]

def recipients_for(brand):
    if not IS_SCHEDULED:
        return list(TEST_RECIPIENTS)
    out = _split_emails(os.environ.get(brand.recipients_variable, "")) or list(SCHEDULED_RECIPIENTS.get(brand.name, []))
    bad = [e for e in out if "@" not in e]
    assert out and not bad, f"[{brand.name}] check the recipient list -- empty or invalid: {bad or out}"
    return out

def send_email_with_attachments(recipients, subject, body, attachment_paths, bcc=False):
    sender = os.environ.get("GMAIL_ADDRESS") or input("Sending Gmail address: ")
    app_password = os.environ.get("GMAIL_APP_PASSWORD") or getpass.getpass("Gmail App Password (not your normal password): ")
    msg = MIMEMultipart()
    msg["From"], msg["Subject"] = sender, subject
    msg["To"] = sender if bcc else ", ".join(recipients)
    msg.attach(MIMEText(body, "plain"))
    for path in attachment_paths:
        mime = ("vnd.openxmlformats-officedocument.spreadsheetml.sheet" if str(path).endswith(".xlsx")
                else "vnd.openxmlformats-officedocument.wordprocessingml.document")
        with open(path, "rb") as f:
            part = MIMEBase("application", mime)
            part.set_payload(f.read())
        encoders.encode_base64(part)
        part.add_header("Content-Disposition", f'attachment; filename="{os.path.basename(path)}"')
        msg.attach(part)
    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(sender, app_password)
        server.send_message(msg, from_addr=sender, to_addrs=list(recipients))

def _glance_line(label, d, col, fmt):
    cur = d[col].iloc[-1]
    pct = pct_change(cur, d[col].iloc[-2]) if len(d) > 1 else None
    change = "" if pct is None else f" ({pct_label(pct)})"
    return f"  • {label}: {'—' if pd.isna(cur) else fmt(cur)}{change}"

def build_email(brand, d):
    """Subject + body of one brand's monthly mail -- edit the wording here."""
    subject = brand.file_stem()                     # e.g. OLJ.August.2026.Matomo -- same as the attachment's name
    glance = "\n".join(_glance_line(label, d, col, fmt) for label, col, fmt in SUMMARY_ROWS)
    vs = f" (vs {PREV_MONTH:%B})" if PREV_MONTH else ""
    body = f"""Hello from Direction Numérique 👋

The {brand.name} {REPORT_NAME} for {REPORT_MONTH:%B %Y} is attached (Word), with every country's visits and share of
visits in Excel. It covers {MONTHS[0]:%b %Y} – {REPORT_MONTH:%b %Y}.

{REPORT_MONTH:%B} at a glance{vs}:
{glance}

🤖 This email is automated and sent on the 1st of every month. Got any questions? Just let us know.

Direction Numérique · L'Orient-Le Jour
"""
    return subject, body

def send_report(brand):
    to = recipients_for(brand)
    subject, body = build_email(brand, BRAND_DATA[brand.name])
    files = [EXPORTED_FILES[brand.name]] + ([EXPORTED_XLSX[brand.name]] if brand.name in EXPORTED_XLSX else [])
    send_email_with_attachments(to, subject, body, files, bcc=EMAIL_AS_BCC)                          # Word + Excel
    print(f"[{brand.name}] emailed to {len(to)} recipient(s): {', '.join(to)}" + ("  (as BCC)" if EMAIL_AS_BCC else ""))

SEND_EMAIL = os.environ.get("SEND_EMAIL", "").strip().lower() in ("1", "true", "yes")
if SEND_EMAIL:
    to_send = [b.name for b in BRANDS]
elif IN_CI:
    to_send = []
else:
    to_send = []
    for b in BRANDS:
        subj, body = build_email(b, BRAND_DATA[b.name])
        print(f"\n--- {b.name} email preview ---\nTo: {', '.join(recipients_for(b))}\nSubject: {subj}\n"
              f"Attachments: {EXPORTED_FILES[b.name]}, {EXPORTED_XLSX.get(b.name, '(no Excel)')}\n\n{body}")
        if input(f"Checked the {b.name} report? Send it now? [y/N] ").strip().lower() == "y":
            to_send.append(b.name)

if to_send and USING_EXAMPLE_DATA:
    raise RuntimeError("Not sending: some numbers are EXAMPLE data (Matomo or GA4 isn't connected -- see steps 3 and 4).")
for name in to_send:
    try:
        send_report(ALL_BRANDS[name])
    except Exception as e:
        print(f"[{name}] email step failed -- paste this error back and I'll adjust.\n{type(e).__name__}: {e}")
if not to_send:
    print("Not sent.")